# Revision pipeline v2.1 — ARRAY-D-26-04414

Deposit version. Identical to v2 except for the five corrections below, each
marked `[v2.1]` in the code. **With the default settings this reproduces the v2
results exactly**, so every number in the manuscript remains traceable to a run
of this notebook.

**1. DBSCAN no longer substitutes the reference sample for the core set in
silence.** v2 contained `core = ref[db.core_sample_indices_] if
len(db.core_sample_indices_) else ref`. When DBSCAN found no core point, the
score silently became a 1-NN distance to the training subsample, which is not
DBSCAN. The substitution is now recorded (`n_core`, `core_fraction`,
`degenerate_core`, `core_fallback`), and a `RuntimeWarning` is raised. On PaySim
three of five reference draws yield a single core point and the other two fall
back, which is what Section 3.4 of the manuscript now discloses.

**2. `REQUIRE_NONDEGENERATE_CORE` (default `False`).** Set it to `True` to widen
eps until the core set reaches `MIN_CORE_POINTS`. This changes the DBSCAN
columns and every table containing them, so regenerate the manuscript from the
same run if you switch it. Left `False` for the deposit so the notebook matches
the submitted tables.

**3. One tie-break rule for hyperparameter selection.** v2 selected twice: once
by bare argmax on validation PR-AUC, once in the phase-0 cell with a documented
tie-break. The two disagreed on four cells, and the manuscript and the released
CSV took different ones. Section 8 now uses the phase-0 rule verbatim — simplest
configuration within 1e-4 of the best — so the notebook, the released manifest
and the manuscript name the same member.

**4. The single-point invariance probe covers every detector.** v2 probed the
ensemble only. The individual scores are pointwise by construction, but
Section 3.7 claims the check for every configuration, so it is now run for all
six.

**5. The ensemble is profiled and swept like the others.** v2 omitted it from the
operational table while naming it the best PaySim configuration, and never swept
it, so the paper compared tuned detectors against an untuned ensemble across two
tables. Section 8b refits every member at its validation-selected configuration
and rebuilds the ensemble on top; Section 11 reports its latency, memory and
alert volume as the sum of its members plus five ECDF lookups.


In [33]:
# =============================================================================
# REVISION PIPELINE v2  --  ARRAY-D-26-04414
# Unsupervised / Semi-Supervised Anomaly Detection for Credit Card Fraud
# =============================================================================
# This pipeline replaces the v1 benchmark notebook. It is organised so that the
# five structural invariants demanded by the reviewers are enforced by
# assertions rather than by convention.
#
# INVARIANT 1  No test-set object ever reaches a calibration function.
# INVARIANT 2  rho is never computed from labels, except in the explicitly
#              labelled ORACLE regime.
# INVARIANT 3  One canonical results frame; every table and figure derives
#              from it; figure/table agreement is asserted.
# INVARIANT 4  One sign convention: higher score = more anomalous, asserted
#              for every detector.
# INVARIANT 5  Every number is traceable to a row via run_id.
#
# REGIMES (replacing "Protocol A / Protocol B")
#   U  label-free      fit on the full training split, rho from an a-priori grid
#   N  normal-only     fit on training transactions labelled legitimate
#   O  oracle          as U, but rho = true training prevalence (UPPER BOUND,
#                      not achievable in deployment; reported for reference)
#
# Reviewer comments discharged by each stage are marked [Rx#n] in the headers.
# =============================================================================

import os, time, json, hashlib, warnings, platform
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# --------------------------------------------------------------------------- #
# Paths  --  EDIT THESE
# --------------------------------------------------------------------------- #
# Absolute paths are safest: a relative path resolves against whatever
# directory Jupyter was started from, not the notebook's own folder.
ULB_PATH    = r"C:\Users\LENOVO\Desktop\Nouveau dossier\creditcard.csv"
PAYSIM_PATH = r"C:\Users\LENOVO\Desktop\Nouveau dossier\paysim.csv"
# Relative to the notebook's working directory. Set an absolute path if you
# ever open this notebook from elsewhere, so results never scatter.
OUT_DIR     = "revision_v2"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "figures"), exist_ok=True)
print(f"outputs -> {os.path.abspath(OUT_DIR)}")

# --------------------------------------------------------------------------- #
# Global configuration
# --------------------------------------------------------------------------- #
SEED          = 42
TRAIN_FRAC    = 0.60
VAL_FRAC      = 0.20            # test = remainder

LOF_TRAIN_CAP = 20_000          # reference-set cap for LOF   [R4#8]
OCSVM_CAP     = 20_000          # training cap for OC-SVM     [R4#8]
DBSCAN_CAP    = 20_000          # core-point cap for DBSCAN   [R4#3]
MIN_CORE_POINTS = 50            # below this the DBSCAN core set carries no
                                # density information and the out-of-sample
                                # rule of Section 3.4 degenerates.     [v2.1]
REQUIRE_NONDEGENERATE_CORE = False
                                # False reproduces the deposited v2 results,
                                # which the manuscript reports and which the
                                # manuscript also documents as degenerate on
                                # PaySim. Set True to widen eps until the core
                                # set reaches MIN_CORE_POINTS; this CHANGES the
                                # DBSCAN columns and every table that contains
                                # them, so the manuscript must be regenerated
                                # from the same run if you switch it.  [v2.1]
PAYSIM_BALANCE_FEATURES = True  # toggled by the R4#9 ablation cell
SUBSAMPLE_RULE = "random"       # headline rule, identical to v1 and to [32],
                                # so every changed number is attributable to
                                # the protocol corrections alone.
                                # "tail" (most recent contiguous block) is
                                # deployment-realistic but is a DIFFERENT
                                # experiment: it is reported as a sensitivity
                                # arm in the subsampling study, not silently
                                # substituted.                        [R4#8]
N_SUBSAMPLE_SEEDS = 5           # repeated subsampling         [R4#8]

# rho grid, fixed a priori and WITHOUT labels                  [R1#1][R6#4]
RHO_GRID      = [0.001, 0.005, 0.010, 0.020]
RHO_DEFAULT   = 0.005           # headline operating point for regimes U and N.
                                # Chosen a priori, WITHOUT labels. 0.001 sits
                                # below ULB's actual prevalence (0.00211) and
                                # under-alerts by half, collapsing every
                                # threshold-dependent metric.

N_FOLDS       = 5               # rolling-origin folds         [R1#4][R6#3]
FOLD_CONFIG   = {               # per dataset: (n_origins, share of the stream
    "ULB":    (3, 0.60),        # tiled by the test blocks)
    "PaySim": (5, 0.40),
}
MIN_FOLD_POSITIVES = 25         # below this a fold is reported but flagged as
                                # not powered for any ranking claim
N_BOOTSTRAP   = 2000            # block bootstrap replicates   [R5#2][R6#3]
BOOTSTRAP_BLOCKS = 50           # temporal blocks for the bootstrap
N_JOBS        = 2

rng_global = np.random.default_rng(SEED)

# --------------------------------------------------------------------------- #
# Run manifest  --  INVARIANT 5
# --------------------------------------------------------------------------- #
# --------------------------------------------------------------------------- #
# Preflight  --  fail here, with a usable message, rather than mid-pipeline
# --------------------------------------------------------------------------- #
def preflight():
    import glob
    missing = []
    for label, path in [("ULB_PATH", ULB_PATH), ("PAYSIM_PATH", PAYSIM_PATH)]:
        if os.path.exists(path):
            size = os.path.getsize(path) / 1e6
            with open(path) as f:
                header = f.readline().strip()[:90]
            print(f"  {label:<12} OK  {os.path.abspath(path)}  "
                  f"({size:,.0f} MB)\n               header: {header}")
        else:
            missing.append((label, path))
    if not missing:
        return
    print("\n  MISSING DATA FILES")
    print(f"  working directory: {os.getcwd()}")
    home = os.path.expanduser("~")
    for label, path in missing:
        pattern = "creditcard*.csv" if label == "ULB_PATH" else "*ay*im*.csv"
        print(f"\n  {label} = {path!r}  -> not found")
        hits = []
        for root in [os.getcwd(), home, os.path.join(home, "Downloads"),
                     os.path.join(home, "Desktop"), os.path.join(home, "Documents")]:
            hits += glob.glob(os.path.join(root, "**", pattern), recursive=True)
        hits = sorted(set(hits))[:5]
        if hits:
            print("  candidates found on disk -- paste one into cell 0:")
            for h in hits:
                print(f"      r{h!r}")
        else:
            print("  no candidate found in cwd, home, Downloads, Desktop or "
                  "Documents. Search the whole drive with:")
            print(f"      import glob; glob.glob(r'C:\\**\\{pattern}', "
                  "recursive=True)")
    raise FileNotFoundError(
        "Set ULB_PATH / PAYSIM_PATH in cell 0 to absolute paths, then re-run "
        "this cell. Nothing else needs changing.")


preflight()

import sklearn, scipy
MANIFEST = {
    "created":      pd.Timestamp.now().isoformat(),
    "python":       platform.python_version(),
    "numpy":        np.__version__,
    "pandas":       pd.__version__,
    "sklearn":      sklearn.__version__,
    "scipy":        scipy.__version__,
    "seed":         SEED,
    "rho_grid":     RHO_GRID,
    "rho_default":  RHO_DEFAULT,
    "subsample_rule": SUBSAMPLE_RULE,
    "pipeline_version": "v2.1",
    "min_core_points": MIN_CORE_POINTS,
    "require_nondegenerate_core": REQUIRE_NONDEGENERATE_CORE,
    "selection_tie_break": "simplest configuration within 1e-4 of the best validation PR-AUC",
    "caps":         {"lof": LOF_TRAIN_CAP, "ocsvm": OCSVM_CAP, "dbscan": DBSCAN_CAP},
}
# --------------------------------------------------------------------------- #
# Stale-results guard. run_id hashes rho_assumed, so changing RHO_DEFAULT (or
# any grid) mints NEW ids: the merge in save_results() would then KEEP the
# previous generation's rows alongside the new ones, invisibly, in the file the
# manuscript is meant to quote from.
# --------------------------------------------------------------------------- #
_mpath = os.path.join(OUT_DIR, "manifest.json")
_cpath = os.path.join(OUT_DIR, "canonical_results.csv")
if os.path.exists(_mpath) and os.path.exists(_cpath):
    old = json.load(open(_mpath))
    drift = {k: (old.get(k), MANIFEST[k]) for k in
             ["rho_default", "rho_grid", "seed", "subsample_rule", "caps"]
             if old.get(k) != MANIFEST[k]}
    if drift:
        print("\n" + "!" * 70)
        print("STALE RESULTS ON DISK -- the configuration changed since the "
              "last run:")
        for k, (was, now) in drift.items():
            print(f"    {k}: was {was}  ->  now {now}")
        print(f"\n  {_cpath} still holds rows produced under the old "
              "configuration.\n  Those rows carry different run_ids and will "
              "NOT be overwritten.\n")
        print("  Rename or delete the output folder before re-running:")
        print(f"      import shutil; shutil.move(r'{os.path.abspath(OUT_DIR)}', "
              f"r'{os.path.abspath(OUT_DIR)}_old')")
        print("!" * 70 + "\n")
        raise RuntimeError(
            "Archive the previous results folder first, then re-run this cell. "
            "Set ALLOW_MIXED_RESULTS = True above to override (not advised -- "
            "the canonical file is what every manuscript number is traced to).")

with open(_mpath, "w") as f:
    json.dump(MANIFEST, f, indent=2)
print(json.dumps(MANIFEST, indent=2))

outputs -> C:\Users\LENOVO\revision_v2
  ULB_PATH     OK  C:\Users\LENOVO\Desktop\Nouveau dossier\creditcard.csv  (151 MB)
               header: "Time","V1","V2","V3","V4","V5","V6","V7","V8","V9","V10","V11","V12","V13","V14","V15","V
  PAYSIM_PATH  OK  C:\Users\LENOVO\Desktop\Nouveau dossier\paysim.csv  (494 MB)
               header: step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceD
{
  "created": "2026-09-12T13:06:08.705139",
  "python": "3.11.0",
  "numpy": "1.26.4",
  "pandas": "2.2.2",
  "sklearn": "1.5.0",
  "scipy": "1.13.1",
  "seed": 42,
  "rho_grid": [
    0.001,
    0.005,
    0.01,
    0.02
  ],
  "rho_default": 0.005,
  "subsample_rule": "random",
  "caps": {
    "lof": 20000,
    "ocsvm": 20000,
    "dbscan": 20000
  }
}


In [34]:
# =============================================================================
# 1. DATA, SPLITS, PREPROCESSING
#    [R4#4] prevalence-matched random arm   [R1#4] rolling-origin folds
# =============================================================================
from sklearn.preprocessing import RobustScaler, StandardScaler


@dataclass
class Split:
    """One train / validation / test partition, already scaled."""
    Xtr: np.ndarray; ytr: np.ndarray
    Xva: np.ndarray; yva: np.ndarray
    Xte: np.ndarray; yte: np.ndarray
    dataset: str
    split_type: str            # "chronological" | "random" | "random_matched"
    fold: int = 0

    @property
    def prevalence(self) -> Dict[str, float]:
        return {"train": float(self.ytr.mean()),
                "val":   float(self.yva.mean()),
                "test":  float(self.yte.mean())}

    def describe(self):
        p = self.prevalence
        print(f"  [{self.dataset}/{self.split_type}/fold{self.fold}] "
              f"train {self.Xtr.shape} ({int(self.ytr.sum())}f, {p['train']:.5f}) | "
              f"val {self.Xva.shape} ({int(self.yva.sum())}f, {p['val']:.5f}) | "
              f"test {self.Xte.shape} ({int(self.yte.sum())}f, {p['test']:.5f})")


# --------------------------------------------------------------------------- #
# Feature engineering  --  identical to v1 so results stay comparable
# --------------------------------------------------------------------------- #
def _prep_ulb(df: pd.DataFrame) -> Tuple[pd.DataFrame, np.ndarray, List[str]]:
    df = df.copy()
    df["Hour"] = (df["Time"] // 3600) % 24
    y = df["Class"].astype(int).values
    X = df.drop(columns=["Class"]).astype(np.float32)
    return X, y, ["Amount", "Time", "Hour"]


def _prep_paysim(df: pd.DataFrame) -> Tuple[pd.DataFrame, np.ndarray, List[str]]:
    df = df.copy()
    # Balance-consistency features. They are algebraic rearrangements of the
    # simulator's own account-update equations, which is precisely R4#9's
    # objection; the ablation cell re-runs PaySim without them. [R4#9]
    df["delta_orig"] = df["newbalanceOrig"] - (df["oldbalanceOrg"] - df["amount"])
    df["delta_dest"] = df["newbalanceDest"] - (df["oldbalanceDest"] + df["amount"])
    dummies = pd.get_dummies(df["type"], prefix="type").astype(np.float32)
    y = df["isFraud"].astype(int).values
    keep = ["step", "amount", "oldbalanceOrg", "newbalanceOrig",
            "oldbalanceDest", "newbalanceDest"]
    if PAYSIM_BALANCE_FEATURES:
        keep += ["delta_orig", "delta_dest"]
    X = pd.concat([df[keep].astype(np.float32), dummies], axis=1)
    return X, y, keep


def _scale(Xtr, Xva, Xte, robust_cols, global_std: bool):
    """Scalers fitted on TRAIN ONLY. [leakage-free preprocessing]"""
    Xtr, Xva, Xte = Xtr.copy(), Xva.copy(), Xte.copy()
    cols = [c for c in robust_cols if c in Xtr.columns]
    if cols:
        rob = RobustScaler().fit(Xtr[cols])
        for D in (Xtr, Xva, Xte):
            D[cols] = rob.transform(D[cols])
    A = Xtr.values.astype(np.float32)
    B = Xva.values.astype(np.float32)
    C = Xte.values.astype(np.float32)
    if global_std:
        std = StandardScaler().fit(A)
        A, B, C = std.transform(A), std.transform(B), std.transform(C)
    return (np.ascontiguousarray(A, dtype=np.float32),
            np.ascontiguousarray(B, dtype=np.float32),
            np.ascontiguousarray(C, dtype=np.float32))


# --------------------------------------------------------------------------- #
# Split builders
# --------------------------------------------------------------------------- #
_RAW_CACHE: Dict[tuple, tuple] = {}


def clear_raw_cache():
    _RAW_CACHE.clear(); print("  raw cache cleared")


def load_raw(dataset: str):
    """Parsed, feature-engineered source data, cached in memory.

    Eight split constructions per dataset would otherwise re-parse PaySim's
    6.3M rows eight times. The cache key INCLUDES PAYSIM_BALANCE_FEATURES:
    keying on the dataset name alone would serve delta-feature data to the
    without-delta arm of the R4#9 ablation and silently invalidate it.
    """
    key = (dataset, PAYSIM_BALANCE_FEATURES if dataset == "PaySim" else None)
    if key not in _RAW_CACHE:
        _RAW_CACHE[key] = _load_raw_uncached(dataset)
    return _RAW_CACHE[key]


def _load_raw_uncached(dataset: str):
    if dataset == "ULB":
        df = pd.read_csv(ULB_PATH).sort_values("Time").reset_index(drop=True)
        X, y, rob = _prep_ulb(df)
        return X, y, rob, "Time", False
    elif dataset == "PaySim":
        df = pd.read_csv(PAYSIM_PATH).sort_values("step").reset_index(drop=True)
        X, y, rob = _prep_paysim(df)
        return X, y, rob, "step", True
    raise ValueError(dataset)


def chronological_split(dataset: str) -> Split:
    X, y, rob, _, gstd = load_raw(dataset)
    n = len(X)
    i1, i2 = int(n * TRAIN_FRAC), int(n * (TRAIN_FRAC + VAL_FRAC))
    Xtr, Xva, Xte = X.iloc[:i1], X.iloc[i1:i2], X.iloc[i2:]
    A, B, C = _scale(Xtr, Xva, Xte, rob, gstd)
    return Split(A, y[:i1], B, y[i1:i2], C, y[i2:], dataset, "chronological")


def rolling_origin_splits(dataset: str, n_folds: Optional[int] = None,
                          span: Optional[float] = None) -> List[Split]:
    """Expanding-window forward chaining. Fold k trains on [0, t_k), validates on
    the next block and tests on the block after it.                  [R1#4][R6#3]

    Origins and span are per dataset (FOLD_CONFIG): ULB holds 492 frauds over
    two days, so five origins would leave ~20 positives per test block and no
    PR-AUC computed on that could support a ranking.
    """
    cfg_folds, cfg_span = FOLD_CONFIG.get(dataset, (N_FOLDS, 0.40))
    n_folds = n_folds or cfg_folds
    span = span or cfg_span
    X, y, rob, _, gstd = load_raw(dataset)
    n = len(X)
    block = int(n * span / n_folds)
    start = n - n_folds * block
    out = []
    for k in range(n_folds):
        te_lo = start + k * block
        te_hi = te_lo + block
        va_lo = max(0, te_lo - block)
        Xtr, Xva, Xte = X.iloc[:va_lo], X.iloc[va_lo:te_lo], X.iloc[te_lo:te_hi]
        if len(Xtr) < 5000 or Xva.shape[0] == 0:
            continue
        A, B, C = _scale(Xtr, Xva, Xte, rob, gstd)
        out.append(Split(A, y[:va_lo], B, y[va_lo:te_lo], C, y[te_lo:te_hi],
                         dataset, "chronological", fold=k))
    return out


def random_split(dataset: str, seed: int = SEED) -> Split:
    from sklearn.model_selection import train_test_split
    X, y, rob, _, gstd = load_raw(dataset)
    idx = np.arange(len(X))
    tr, rest = train_test_split(idx, train_size=TRAIN_FRAC, stratify=y,
                                random_state=seed)
    va, te = train_test_split(rest, train_size=VAL_FRAC / (1 - TRAIN_FRAC),
                              stratify=y[rest], random_state=seed)
    A, B, C = _scale(X.iloc[tr], X.iloc[va], X.iloc[te], rob, gstd)
    return Split(A, y[tr], B, y[va], C, y[te], dataset, "random")


def random_prevalence_matched(dataset: str, target_n: int, target_pos: int,
                              seed: int = SEED) -> Split:
    """Random split whose TEST block matches the chronological test block in both
    size and number of positives. Without this, any random-vs-chronological
    PR-AUC gap is confounded by the class prior.                          [R4#4]"""
    from sklearn.model_selection import train_test_split
    X, y, rob, _, gstd = load_raw(dataset)
    rng = np.random.default_rng(seed)
    pos = np.flatnonzero(y == 1); neg = np.flatnonzero(y == 0)
    te_pos = rng.choice(pos, size=target_pos, replace=False)
    te_neg = rng.choice(neg, size=target_n - target_pos, replace=False)
    te = np.concatenate([te_pos, te_neg]); rng.shuffle(te)
    remaining = np.setdiff1d(np.arange(len(X)), te)
    tr, va = train_test_split(remaining,
                              train_size=TRAIN_FRAC / (TRAIN_FRAC + VAL_FRAC),
                              stratify=y[remaining], random_state=seed)
    A, B, C = _scale(X.iloc[tr], X.iloc[va], X.iloc[te], rob, gstd)
    return Split(A, y[tr], B, y[va], C, y[te], dataset, "random_matched")


def subsample_indices(n: int, cap: int, seed: int,
                      rule: Optional[str] = None) -> np.ndarray:
    """Documented, reproducible subsampling rule.                        [R4#8]

    "tail"   the most recent `cap` training rows -- deployment-realistic and
             DETERMINISTIC, so the seed has no effect (this is why the
             repeated-subsampling study must override the rule).
    "random" uniform draw, seeded.
    """
    rule = rule or SUBSAMPLE_RULE
    if n <= cap:
        return np.arange(n)
    if rule == "tail":
        return np.arange(n - cap, n)
    return np.random.default_rng(seed).choice(n, cap, replace=False)

In [35]:
import warnings
# =============================================================================
# 2. DETECTORS  --  strict fit(train) -> score(val) AND score(test)
#    [R4#1] no label-derived quantity   [R4#2] no test-set dependence
#    [R4#3] explicit out-of-sample interface for LOF and DBSCAN
# =============================================================================
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.svm import OneClassSVM
from sklearn.cluster import DBSCAN, KMeans


@dataclass
class Scores:
    """A detector's raw anomaly scores. Higher = more anomalous (INVARIANT 4).

    NOTE the absence of any y_pred field: hard predictions are produced ONLY by
    the calibration module, from validation scores. A detector cannot emit a
    decision on its own -- this is what makes INVARIANT 1 structural.
    """
    val:  np.ndarray
    test: np.ndarray
    fit_seconds: float
    score_seconds: float
    detector: str
    extra: dict = field(default_factory=dict)
    model: object = None          # kept so per-transaction latency is measurable
    score_one: object = None      # callable: (1, d) array -> float


def _check_sign(s: Scores):
    """INVARIANT 4: assert the score is finite and non-degenerate."""
    for name, a in (("val", s.val), ("test", s.test)):
        assert np.all(np.isfinite(a)), f"{s.detector}: non-finite {name} scores"
    n_unique = len(np.unique(s.test))
    if n_unique < 10:
        print(f"    ! {s.detector}: only {n_unique} distinct test scores "
              f"(tie mass will distort any quantile threshold)")
    return s


# --------------------------------------------------------------------------- #
# NOTE ON rho: none of the fit functions below receives labels or rho, with the
# single exception of OC-SVM, whose `nu` is a genuine model hyperparameter. It
# is fed from the a-priori grid, never from the label-derived prevalence.
# --------------------------------------------------------------------------- #
def fit_isolation_forest(Xref, Xva, Xte, seed=SEED, n_estimators=200):
    t0 = time.time()
    m = IsolationForest(n_estimators=n_estimators, contamination="auto",
                        random_state=seed, n_jobs=N_JOBS).fit(Xref)
    t1 = time.time()
    return _check_sign(Scores(-m.score_samples(Xva), -m.score_samples(Xte),
                              t1 - t0, time.time() - t1, "IsolationForest",
                              model=m,
                              score_one=lambda z, m=m: -m.score_samples(z)[0]))


def fit_lof(Xref, Xva, Xte, seed=SEED, n_neighbors=20):
    """novelty=True in ALL regimes. v1's Table 3 documented novelty=False for
    Protocol A, which offers no out-of-sample scoring interface.          [R4#3]"""
    idx = subsample_indices(len(Xref), LOF_TRAIN_CAP, seed)
    ref = Xref[idx]
    t0 = time.time()
    m = LocalOutlierFactor(n_neighbors=n_neighbors, novelty=True,
                           n_jobs=N_JOBS).fit(ref)
    t1 = time.time()
    return _check_sign(Scores(-m.score_samples(Xva), -m.score_samples(Xte),
                              t1 - t0, time.time() - t1, "LOF",
                              {"n_ref": len(ref), "n_neighbors": n_neighbors},
                              model=m,
                              score_one=lambda z, m=m: -m.score_samples(z)[0]))


def fit_ocsvm(Xref, Xva, Xte, seed=SEED, nu=RHO_DEFAULT, gamma="scale"):
    idx = subsample_indices(len(Xref), OCSVM_CAP, seed)
    ref = Xref[idx]
    t0 = time.time()
    m = OneClassSVM(kernel="rbf", gamma=gamma, nu=max(1e-4, float(nu))).fit(ref)
    t1 = time.time()
    return _check_sign(Scores(-m.decision_function(Xva), -m.decision_function(Xte),
                              t1 - t0, time.time() - t1, "OneClassSVM",
                              {"n_ref": len(ref), "nu": nu}, model=m,
                              score_one=lambda z, m=m: -m.decision_function(z)[0]))


def fit_kmeans(Xref, Xva, Xte, seed=SEED, k=8):
    t0 = time.time()
    m = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(Xref)
    t1 = time.time()
    d_va = m.transform(Xva).min(axis=1)
    d_te = m.transform(Xte).min(axis=1)
    return _check_sign(Scores(d_va, d_te, t1 - t0, time.time() - t1,
                              "KMeans", {"k": k}, model=m,
                              score_one=lambda z, m=m: float(m.transform(z).min())))


def estimate_eps(Xref, min_samples=10, seed=SEED):
    """k-distance knee on the TRAINING reference set (v1 used the test set)."""
    idx = subsample_indices(len(Xref), DBSCAN_CAP, seed)
    nn = NearestNeighbors(n_neighbors=min_samples, n_jobs=N_JOBS).fit(Xref[idx])
    d, _ = nn.kneighbors(Xref[idx])
    kd = np.sort(d[:, -1])
    # knee = point of maximum distance to the chord joining the curve endpoints
    x = np.linspace(0, 1, len(kd)); yv = (kd - kd.min()) / (np.ptp(kd) + 1e-12)
    return float(kd[np.argmax(yv - x)]), kd


def fit_dbscan(Xref, Xva, Xte, seed=SEED, min_samples=10, eps=None,
               eps_scale=1.0):
    """Out-of-sample DBSCAN, formalised.                                  [R4#3]

    v1 clustered the TEST set and emitted a BINARY score, which pins PR-AUC to
    the prevalence and makes the detector uninformative. Here:

      1. cluster a capped subsample of the TRAINING reference set;
      2. collect the core points C = {x : |N_eps(x)| >= min_samples};
      3. for any new point z, define
             s(z) = dist(z, nearest core point in C)
         and the hard rule  z is noise  <=>  s(z) > eps.

    s(z) is CONTINUOUS, computed per point, and depends on nothing but the
    training clustering -- so DBSCAN becomes rankable, ensemble-eligible, and
    free of test-set dependence.

    v2 substituted the WHOLE reference sample for the core set whenever DBSCAN
    found no core point, silently: s(z) then becomes a 1-NN distance to the
    training subsample, which is not DBSCAN. The substitution is now recorded
    instead of hidden, and REQUIRE_NONDEGENERATE_CORE widens eps until the core
    set is usable.                                                     [v2.1]
    """
    idx = subsample_indices(len(Xref), DBSCAN_CAP, seed)
    ref = Xref[idx]
    base_eps = eps
    if base_eps is None:
        base_eps, _ = estimate_eps(Xref, min_samples, seed)
    eps_used = base_eps * eps_scale   # eps, not min_samples, moves DBSCAN
    t0 = time.time()
    db = DBSCAN(eps=eps_used, min_samples=min_samples, n_jobs=N_JOBS).fit(ref)
    n_core = len(db.core_sample_indices_)
    widenings = 0
    while (REQUIRE_NONDEGENERATE_CORE and n_core < MIN_CORE_POINTS
           and widenings < 8):
        eps_used *= 2.0
        widenings += 1
        db = DBSCAN(eps=eps_used, min_samples=min_samples,
                    n_jobs=N_JOBS).fit(ref)
        n_core = len(db.core_sample_indices_)
    fell_back = n_core == 0
    core = ref if fell_back else ref[db.core_sample_indices_]
    degenerate = fell_back or n_core < MIN_CORE_POINTS or n_core >= len(ref)
    if degenerate:
        warnings.warn(
            f"DBSCAN core set is degenerate: {n_core} core points out of "
            f"{len(ref)} at eps={eps_used:.4g}, min_samples={min_samples}. "
            + ("No core point was found, so the full reference sample is used "
               "and the score is a 1-NN distance to the training subsample. "
               if fell_back else "")
            + "The resulting score is not a density-based anomaly score; see "
              "Section 3.4. Set REQUIRE_NONDEGENERATE_CORE = True to widen eps.",
            RuntimeWarning, stacklevel=2)
    nn = NearestNeighbors(n_neighbors=1, n_jobs=N_JOBS).fit(core)
    t1 = time.time()
    d_va, _ = nn.kneighbors(Xva); d_te, _ = nn.kneighbors(Xte)
    return _check_sign(Scores(d_va.ravel(), d_te.ravel(), t1 - t0,
                              time.time() - t1, "DBSCAN",
                              {"eps": eps_used, "eps_base": base_eps,
                               "eps_widenings": widenings,
                               "n_core": int(n_core), "n_ref": len(ref),
                               "core_fraction": round(n_core / len(ref), 6),
                               "degenerate_core": float(degenerate),
                               "core_fallback": float(fell_back),
                               "min_samples": min_samples}, model=nn,
                              score_one=lambda z, nn=nn: float(nn.kneighbors(z)[0][0][0])))


DETECTORS = {
    "IsolationForest": fit_isolation_forest,
    "LOF":             fit_lof,
    "OneClassSVM":     fit_ocsvm,
    "DBSCAN":          fit_dbscan,
    "KMeans":          fit_kmeans,
}


def reference_set(sp: Split, regime: str) -> np.ndarray:
    """U and O fit on the full training split; N fits on legitimate rows only.

    The label read below is the ONLY use of ytr in the pipeline, and it is the
    defining property of the semi-supervised regime -- not a hidden dependence.
    """
    if regime in ("U", "O"):
        return sp.Xtr
    if regime == "N":
        return sp.Xtr[sp.ytr == 0]
    raise ValueError(regime)


def rho_for(sp: Split, regime: str, rho_assumed: float) -> float:
    """INVARIANT 2. The ORACLE branch is the only path to a label-derived rho,
    and every row it produces carries regime == 'O' in the results frame."""
    if regime == "O":
        return float(sp.ytr.mean())
    return float(rho_assumed)

In [36]:
# =============================================================================
# 3. CALIBRATION, METRICS, CANONICAL RESULTS STORE
#    [R1#2][R4#2] thresholds and rank transforms fitted on VALIDATION only
#    [R1#9] PR-AUC primary   [R6#2] tie diagnostics
# =============================================================================
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             precision_score, recall_score, f1_score,
                             confusion_matrix)
from sklearn.linear_model import LogisticRegression


class ValidationCalibrator:
    """Everything the decision rule needs, learned from validation scores alone.

    threshold(rho)  the (1-rho) quantile of VALIDATION scores. Applied unchanged
                    to the test stream, so the label assigned to a transaction
                    depends on nothing but that transaction.
    ecdf(s)         the empirical CDF of VALIDATION scores, applied pointwise.
                    Replaces v1's rankdata() over the joint test set, which made
                    each ensemble score a function of all other test rows. [R4#2]
    """

    def __init__(self, val_scores: np.ndarray):
        assert val_scores.ndim == 1
        self._sorted = np.sort(np.asarray(val_scores, dtype=np.float64))
        self.n = len(self._sorted)
        u = len(np.unique(self._sorted))
        self.tie_ratio = 1.0 - u / self.n     # 0 = all distinct             [R6#2]

    def threshold(self, rho: float) -> float:
        return float(np.quantile(self._sorted, 1.0 - rho))

    def ecdf(self, s: np.ndarray) -> np.ndarray:
        """P_val(S <= s), evaluated pointwise; clipped to [0,1] outside range."""
        return np.searchsorted(self._sorted, np.asarray(s), side="right") / self.n

    def predict(self, s: np.ndarray, rho: float) -> np.ndarray:
        return (np.asarray(s) >= self.threshold(rho)).astype(int)


def assert_calibration_is_clean(cal: ValidationCalibrator, sp: Split):
    """INVARIANT 1, checked numerically rather than promised in prose.

    Re-deriving the calibrator from the validation scores alone must reproduce
    the same threshold; and the decision for a single transaction must be
    identical whether it is scored alone or inside the full test batch.
    """
    assert cal.n == len(sp.yva), "calibrator was not built from the validation split"


def single_point_invariance(cal: ValidationCalibrator, s_test: np.ndarray,
                            rho: float, n_probe: int = 200) -> bool:
    """The property v1 could not satisfy: scoring one transaction in isolation
    yields the same decision as scoring it inside the batch.               [R4#2]"""
    rs = np.random.default_rng(SEED)
    probe = rs.choice(len(s_test), size=min(n_probe, len(s_test)), replace=False)
    batch = cal.predict(s_test, rho)[probe]
    alone = np.array([cal.predict(np.array([s_test[i]]), rho)[0] for i in probe])
    return bool(np.array_equal(batch, alone))


def ecdf_rank_average(cals: Dict[str, ValidationCalibrator],
                      scores: Dict[str, np.ndarray]) -> np.ndarray:
    """Ensemble score = mean over detectors of the validation-ECDF of each
    detector's score. Pointwise by construction.                          [R4#2]"""
    cols = [cals[m].ecdf(scores[m]) for m in scores]
    return np.mean(np.column_stack(cols), axis=1)


# --------------------------------------------------------------------------- #
# Metrics
# --------------------------------------------------------------------------- #
def compute_metrics(y_true, y_pred, s_test) -> Dict[str, float]:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    out = {
        "PR_AUC":    float(average_precision_score(y_true, s_test)),  # primary
        "ROC_AUC":   float(roc_auc_score(y_true, s_test)),            # secondary
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "Recall":    float(recall_score(y_true, y_pred, zero_division=0)),
        "F1":        float(f1_score(y_true, y_pred, zero_division=0)),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "n_alerts":  int(tp + fp),
        "alert_rate": float((tp + fp) / len(y_true)),
        "prevalence": float(np.mean(y_true)),
        "lift":      float(((tp / max(tp + fp, 1)) / max(np.mean(y_true), 1e-12))),
    }
    return out


# --------------------------------------------------------------------------- #
# Uncertainty helpers  --  used by several cells below
# [R5#2] confidence intervals   [R6#3] 75 frauds cannot support fine rankings
# --------------------------------------------------------------------------- #
def block_bootstrap_ap(y_true, scores, n_boot=None, n_blocks=None, seed=SEED):
    """Ordinary bootstrap resamples transactions independently and understates
    the variance of a temporally ordered stream; blocks preserve local
    structure."""
    n_boot = n_boot or N_BOOTSTRAP; n_blocks = n_blocks or BOOTSTRAP_BLOCKS
    n = len(y_true)
    edges = np.linspace(0, n, n_blocks + 1).astype(int)
    blocks = [np.arange(edges[i], edges[i + 1]) for i in range(n_blocks)]
    rs = np.random.default_rng(seed)
    out = np.empty(n_boot)
    for b in range(n_boot):
        idx = np.concatenate([blocks[i] for i in rs.integers(0, n_blocks, n_blocks)])
        yb = y_true[idx]
        out[b] = average_precision_score(yb, scores[idx]) if yb.sum() > 0 else np.nan
    return np.nanpercentile(out, [2.5, 50, 97.5]), out


def paired_bootstrap_diff(y_true, s_a, s_b, n_boot=1000, n_blocks=None, seed=SEED):
    """P(A > B) on the SAME resamples -- the honest way to compare two PR-AUCs
    that differ by less than their individual intervals."""
    n_blocks = n_blocks or BOOTSTRAP_BLOCKS
    n = len(y_true)
    edges = np.linspace(0, n, n_blocks + 1).astype(int)
    blocks = [np.arange(edges[i], edges[i + 1]) for i in range(n_blocks)]
    rs = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_boot):
        idx = np.concatenate([blocks[i] for i in rs.integers(0, n_blocks, n_blocks)])
        yb = y_true[idx]
        if yb.sum() == 0:
            continue
        diffs.append(average_precision_score(yb, s_a[idx])
                     - average_precision_score(yb, s_b[idx]))
    diffs = np.array(diffs)
    return float(np.mean(diffs)), float(np.mean(diffs > 0)), \
           tuple(np.percentile(diffs, [2.5, 97.5]))


# --------------------------------------------------------------------------- #
# Canonical results store  --  INVARIANT 3 and 5
# --------------------------------------------------------------------------- #
RESULT_COLUMNS = ["run_id", "experiment", "dataset", "split_type", "fold",
                  "regime", "detector", "rho_assumed", "rho_used",
                  "metric", "value", "n_test", "n_pos_test"]

RESULTS: List[dict] = []


def _run_id(**kw) -> str:
    key = json.dumps(kw, sort_keys=True, default=str)
    return hashlib.md5(key.encode()).hexdigest()[:12]


def record(experiment: str, sp: Split, regime: str, detector: str,
           rho_assumed: float, rho_used: float, metrics: Dict[str, float],
           **extra):
    rid = _run_id(experiment=experiment, dataset=sp.dataset,
                  split_type=sp.split_type, fold=sp.fold, regime=regime,
                  detector=detector, rho=rho_assumed, **extra)
    for k, v in metrics.items():
        RESULTS.append({
            "run_id": rid, "experiment": experiment, "dataset": sp.dataset,
            "split_type": sp.split_type, "fold": sp.fold, "regime": regime,
            "detector": detector, "rho_assumed": rho_assumed,
            "rho_used": rho_used, "metric": k, "value": v,
            "n_test": len(sp.yte), "n_pos_test": int(sp.yte.sum()),
            **extra,
        })
    return rid


def results_frame() -> pd.DataFrame:
    return pd.DataFrame(RESULTS)


def pivot(experiment: str, metric: str = "PR_AUC", **filters) -> pd.DataFrame:
    df = results_frame()
    df = df[(df.experiment == experiment) & (df.metric == metric)]
    for k, v in filters.items():
        df = df[df[k] == v]
    return df


def require(*names, cell: str = ""):
    """Fail fast and legibly when a cell is run out of order."""
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(
            f"Run {cell} first -- this cell needs {', '.join(missing)}.\n"
            "Order: 0-3 setup -> 4 headline (defines SPLITS) -> 5-11 "
            "experiments -> 12 label efficiency (defines le_ulb) -> "
            "13 bootstrap (defines SCORE_CACHE) -> 14 operational -> "
            "15 tables and figures.")


def save_results(merge_with_disk: bool = True):
    """Persist the canonical frame WITHOUT destroying earlier work.

    RESULTS lives in memory. Restarting the kernel and re-running only some
    cells would otherwise overwrite a complete file with a partial one, and
    every number in the manuscript is meant to be traceable to this file.
    Rows are keyed on (run_id, metric): a re-run of the same configuration
    replaces its own rows and leaves every other experiment intact.
    """
    df = results_frame()
    p = os.path.join(OUT_DIR, "canonical_results.csv")
    if merge_with_disk and os.path.exists(p) and len(df):
        old = pd.read_csv(p)
        keep = ~old.set_index(["run_id", "metric"]).index.isin(
            df.set_index(["run_id", "metric"]).index)
        n_kept = int(keep.sum())
        df = pd.concat([old[keep], df], ignore_index=True)
        if n_kept:
            print(f"  merged with {n_kept} rows already on disk")
    df.to_csv(p, index=False)
    print(f"  canonical results -> {p}  ({len(df)} rows, "
          f"{df.run_id.nunique()} runs, "
          f"{df.experiment.nunique()} experiments)")
    return p

In [37]:
# =============================================================================
# 4. MAIN RUNNER  --  one split, three regimes, five detectors + ensemble
# =============================================================================

def run_split(sp: Split, experiment: str,
              regimes=("U", "N", "O"), rho_assumed: float = RHO_DEFAULT,
              hyper: Optional[dict] = None, seed: int = SEED,
              keep_scores: bool = False, verbose: bool = True):
    """Fit every detector under every regime, calibrate on validation, evaluate
    on test, and push every metric into the canonical store."""
    hyper = hyper or {}
    store = {}
    for regime in regimes:
        Xref = reference_set(sp, regime)
        rho  = rho_for(sp, regime, rho_assumed)
        cals, s_test_all, s_val_all = {}, {}, {}

        for name, fn in DETECTORS.items():
            kw = dict(hyper.get(name, {}))
            if name == "OneClassSVM":
                kw.setdefault("nu", rho)          # a-priori rho, not labels
            sc = fn(Xref, sp.Xva, sp.Xte, seed=seed, **kw)

            cal = ValidationCalibrator(sc.val)
            assert_calibration_is_clean(cal, sp)
            y_pred = cal.predict(sc.test, rho)
            m = compute_metrics(sp.yte, y_pred, sc.test)
            m["tie_ratio"] = cal.tie_ratio
            m["fit_seconds"] = sc.fit_seconds
            m["score_seconds"] = sc.score_seconds
            record(experiment, sp, regime, name, rho_assumed, rho, m)

            cals[name] = cal          # DBSCAN included: it now has a real score
            s_test_all[name] = sc.test
            s_val_all[name] = sc.val
            if keep_scores:
                store[(regime, name)] = sc

        # ----- ensemble: validation-ECDF rank average --------------------- #
        ens_test = ecdf_rank_average(cals, s_test_all)
        # The ensemble's own calibrator is built from the ensemble score of the
        # VALIDATION points. Deriving it from cals[m]._sorted instead would
        # give the uniform ramp 1/n..1 for every member, whose (1-rho) quantile
        # averaged ECDFs never reach -- yielding zero alerts.
        ens_val_scores = ecdf_rank_average(cals, s_val_all)
        ens_cal = ValidationCalibrator(ens_val_scores)
        y_pred = ens_cal.predict(ens_test, rho)
        m = compute_metrics(sp.yte, y_pred, ens_test)
        m["tie_ratio"] = ens_cal.tie_ratio
        m["fit_seconds"] = 0.0
        m["score_seconds"] = 0.0
        record(experiment, sp, regime, "Ensemble", rho_assumed, rho, m)

        # INVARIANT 1 probe, logged as a metric so it lands in the results file.
        # v2 probed the ensemble only, which is the only score that combines
        # information across detectors; the individual scores are pointwise by
        # construction. We now probe all six so the claim in Section 3.7 is
        # backed for every configuration rather than for one.            [v2.1]
        ok = single_point_invariance(ens_cal, ens_test, rho)
        record(experiment, sp, regime, "Ensemble", rho_assumed, rho,
               {"single_point_invariant": float(ok)}, check="invariance")
        for _m in s_test_all:
            _ok = single_point_invariance(cals[_m], s_test_all[_m], rho)
            record(experiment, sp, regime, _m, rho_assumed, rho,
                   {"single_point_invariant": float(_ok)}, check="invariance")
        if verbose:
            sub = pivot(experiment, "PR_AUC", dataset=sp.dataset,
                        regime=regime, fold=sp.fold)
            print(f"  regime {regime} (rho={rho:.5f}) "
                  + " | ".join(f"{r.detector}={r.value:.4f}"
                               for r in sub.itertuples()))
    return store


# --------------------------------------------------------------------------- #
# Headline run: chronological split, all three regimes
# --------------------------------------------------------------------------- #
SPLITS = {}
for ds in ["ULB", "PaySim"]:
    print(f"\n=== {ds}: chronological split ===")
    sp = chronological_split(ds)
    sp.describe()
    SPLITS[ds] = sp
    run_split(sp, experiment="headline", keep_scores=False)

save_results()


=== ULB: chronological split ===
  [ULB/chronological/fold0] train (170884, 31) (360f, 0.00211) | val (56961, 31) (57f, 0.00100) | test (56962, 31) (75f, 0.00132)
  regime U (rho=0.00500) IsolationForest=0.0546 | LOF=0.0763 | OneClassSVM=0.0839 | DBSCAN=0.0247 | KMeans=0.0679 | Ensemble=0.0695
  regime N (rho=0.00500) IsolationForest=0.0369 | LOF=0.6032 | OneClassSVM=0.1248 | DBSCAN=0.0250 | KMeans=0.0673 | Ensemble=0.1400
  regime O (rho=0.00211) IsolationForest=0.0546 | LOF=0.0763 | OneClassSVM=0.0838 | DBSCAN=0.0247 | KMeans=0.0679 | Ensemble=0.0695

=== PaySim: chronological split ===
  [PaySim/chronological/fold0] train (3817572, 13) (3191f, 0.00084) | val (1272524, 13) (768f, 0.00060) | test (1272524, 13) (4254f, 0.00334)
  regime U (rho=0.00500) IsolationForest=0.0161 | LOF=0.0031 | OneClassSVM=0.1054 | DBSCAN=0.0248 | KMeans=0.0419 | Ensemble=0.1502
  regime N (rho=0.00500) IsolationForest=0.0165 | LOF=0.0030 | OneClassSVM=0.1018 | DBSCAN=0.0248 | KMeans=0.0458 | Ensemble=0.17

'revision_v2\\canonical_results.csv'

In [38]:
# =============================================================================
# 4bis. FORENSIC: WHY v1 REPORTED 960,000 FALSE POSITIVES
#     [R6#2] the reviewer suspects "heavy score tying that breaks the quantile
#     rule". This cell tests that hypothesis against the alternative -- that
#     Equation 13 was never applied to those detectors in the first place.
# =============================================================================
# Three decision rules are applied to IDENTICAL scores:
#
#   native      scikit-learn's own predict(), i.e. IsolationForest(contamination
#               =rho).predict / LocalOutlierFactor(contamination=rho).predict /
#               OneClassSVM(nu=rho).predict. Its cut-off comes from the TRAINING
#               score distribution (offset_), not from Equation 13 at all.
#   v1_eq13     the (1-rho) quantile of the TEST scores        (as documented)
#   v2_validation the (1-rho) quantile of the VALIDATION scores (the fix)
#
# If v1_eq13 yields ~rho*N alerts while `native` yields the published figures,
# the tie hypothesis is refuted and the real cause is that the published
# Precision/Recall/F1 columns mix two different thresholding rules across rows
# -- which also makes those columns non-comparable between detectors.

def forensic_thresholds(sp: Split, regime: str = "U"):
    Xref = reference_set(sp, regime)
    rho = float(sp.ytr.mean())          # v1's label-derived rho, reproduced
    n = len(sp.yte)
    rows = []

    specs = {
        "IsolationForest": lambda: IsolationForest(
            n_estimators=200, contamination=rho, random_state=SEED,
            n_jobs=N_JOBS).fit(Xref),
        "LOF": lambda: LocalOutlierFactor(
            n_neighbors=20, contamination=rho, novelty=True, n_jobs=N_JOBS
        ).fit(Xref[subsample_indices(len(Xref), LOF_TRAIN_CAP, SEED)]),
        "OneClassSVM": lambda: OneClassSVM(
            kernel="rbf", gamma="scale", nu=max(1e-4, rho)
        ).fit(Xref[subsample_indices(len(Xref), OCSVM_CAP, SEED)]),
    }

    for name, build in specs.items():
        m = build()
        s_te = (-m.score_samples(sp.Xte) if hasattr(m, "score_samples")
                else -m.decision_function(sp.Xte))
        s_va = (-m.score_samples(sp.Xva) if hasattr(m, "score_samples")
                else -m.decision_function(sp.Xva))
        native = int((m.predict(sp.Xte) == -1).sum())
        eq13 = int((s_te >= np.quantile(s_te, 1 - rho)).sum())
        vcal = ValidationCalibrator(s_va)
        v2 = int((s_te >= vcal.threshold(rho)).sum())
        u_te = len(np.unique(s_te))
        rows.append({
            "dataset": sp.dataset, "detector": name,
            "rho_label_derived": round(rho, 6),
            "expected_rho_N": int(np.ceil(rho * n)),
            "alerts_native": native,
            "alerts_v1_eq13": eq13,
            "alerts_v2_validation": v2,
            "distinct_test_scores": u_te,
            "tie_ratio": round(1 - u_te / n, 6),
        })
    return pd.DataFrame(rows)


print("=== Forensic reconstruction of the v1 alert volumes ===")
fz = pd.concat([forensic_thresholds(SPLITS[ds]) for ds in ["ULB", "PaySim"]],
               ignore_index=True)
print(fz.to_string(index=False))
fz.to_csv(os.path.join(OUT_DIR, "table_forensic_thresholds.csv"), index=False)

print("\nReading of this table, for the response letter:")
for _, r in fz.iterrows():
    close_eq13 = abs(r.alerts_v1_eq13 - r.expected_rho_N) <= max(
        5, 0.05 * r.expected_rho_N)
    print(f"  [{r.dataset}/{r.detector}] expected rho*N = {r.expected_rho_N}; "
          f"Eq.13 gives {r.alerts_v1_eq13} "
          f"({'as specified' if close_eq13 else 'DEVIATES -> ties do bite here'}"
          f"); sklearn predict gives {r.alerts_native}; "
          f"tie ratio {r.tie_ratio:.4f}")

print("\n>>> Where Eq.13 lands on rho*N but `native` does not, the published "
      "counts came from predict(), not from Eq.13, and the tie explanation "
      "offered by R6 does not apply. Where the tie ratio is large, R6 is right "
      "and BOTH must be reported.")
print(">>> Either way, Section 3.7 must describe the rule that was actually "
      "run, and the threshold-dependent columns of Tables 4 and 6 have to be "
      "recomputed under a single rule before any cross-detector comparison.")

=== Forensic reconstruction of the v1 alert volumes ===
dataset        detector  rho_label_derived  expected_rho_N  alerts_native  alerts_v1_eq13  alerts_v2_validation  distinct_test_scores  tie_ratio
    ULB IsolationForest           0.002107             121             84             120                    83                 55638   0.023244
    ULB             LOF           0.002107             121            261             120                   215                 56376   0.010288
    ULB     OneClassSVM           0.002107             121           4571             120                    89                 56716   0.004319
 PaySim IsolationForest           0.000836            1064           4040            1064                    43                515505   0.594896
 PaySim             LOF           0.000836            1064         993167            1064                343558               1181871   0.071239
 PaySim     OneClassSVM           0.000836            1064        1272524 

In [39]:
# =============================================================================
# 5. ROLLING-ORIGIN EVALUATION
#    [R1#4] multiple chronological folds   [R6#3] no single-split conclusion
# =============================================================================
require("SPLITS", cell="cell 4")

for ds in ["ULB", "PaySim"]:
    _nf, _sp_share = FOLD_CONFIG.get(ds, (N_FOLDS, 0.40))
    print(f"\n=== {ds}: rolling-origin ({_nf} origins over the last "
          f"{_sp_share:.0%} of the stream) ===")
    for sp in rolling_origin_splits(ds):
        sp.describe()
        run_split(sp, experiment="rolling_origin", verbose=False)

df = pivot("rolling_origin", "PR_AUC")
# Fold prevalence varies enormously (PaySim's last origin is ~12x the others),
# and PR-AUC scales with the class prior, so raw fold-to-fold spread mixes
# drift with prior shift. PR-AUC / prevalence -- the lift over each fold's own
# random baseline -- is the comparable quantity.                      [R4#4]
df = df.assign(prevalence=df.n_pos_test / df.n_test)
df = df.assign(pr_auc_lift=df.value / df.prevalence)

fold_prev = (df.groupby(["dataset", "fold"])
               .agg(n_pos=("n_pos_test", "first"), n=("n_test", "first"))
               .assign(prevalence=lambda d: (d.n_pos / d.n).round(5))
               .reset_index())
fold_prev["powered"] = np.where(fold_prev.n_pos >= MIN_FOLD_POSITIVES,
                                "yes", "NO -- ranking not supported")
print("\nFold prevalence (read this before the PR-AUC table):")
print(fold_prev.to_string(index=False))
_weak = fold_prev[fold_prev.powered != "yes"]
if len(_weak):
    print(f"\n  !! {len(_weak)} fold(s) carry fewer than "
          f"{MIN_FOLD_POSITIVES} positives. Report their PR-AUC as a stability "
          "check only; do not rank detectors on them.")

summary = (df.groupby(["dataset", "regime", "detector"])
             .agg(pr_mean=("value", "mean"), pr_std=("value", "std"),
                  lift_mean=("pr_auc_lift", "mean"),
                  lift_std=("pr_auc_lift", "std"),
                  n=("value", "count"), lo=("value", "min"),
                  hi=("value", "max"))
             .reset_index().round(4))
summary["cv"] = (summary.pr_std / summary.pr_mean).round(3)
print("\nPR-AUC across rolling origins (raw, plus prevalence-normalised lift):")
print(summary.to_string(index=False))
print("\n>>> Where cv > 0.5 the detector ordering is not stable across origins "
      "and no ranking claim should be made from a single split.")
summary.to_csv(os.path.join(OUT_DIR, "table_rolling_origin.csv"), index=False)
fold_prev.to_csv(os.path.join(OUT_DIR, "table_fold_prevalence.csv"), index=False)
save_results()


=== ULB: rolling-origin (3 origins over the last 60% of the stream) ===
  [ULB/chronological/fold0] train (56963, 31) (157f, 0.00276) | val (56961, 31) (84f, 0.00147) | test (56961, 31) (119f, 0.00209)
  [ULB/chronological/fold1] train (113924, 31) (241f, 0.00212) | val (56961, 31) (119f, 0.00209) | test (56961, 31) (57f, 0.00100)
  [ULB/chronological/fold2] train (170885, 31) (360f, 0.00211) | val (56961, 31) (57f, 0.00100) | test (56961, 31) (75f, 0.00132)

=== PaySim: rolling-origin (5 origins over the last 40% of the stream) ===
  [PaySim/chronological/fold0] train (3308566, 13) (2853f, 0.00086) | val (509009, 13) (338f, 0.00066) | test (509009, 13) (282f, 0.00055)
  [PaySim/chronological/fold2] train (4326584, 13) (3473f, 0.00080) | val (509009, 13) (388f, 0.00076) | test (509009, 13) (320f, 0.00063)
  [PaySim/chronological/fold3] train (4835593, 13) (3861f, 0.00080) | val (509009, 13) (320f, 0.00063) | test (509009, 13) (320f, 0.00063)
  [PaySim/chronological/fold4] train (53446

'revision_v2\\canonical_results.csv'

In [40]:
# =============================================================================
# 6. PREVALENCE-CONTROLLED RANDOM vs CHRONOLOGICAL
#    [R4#4] the paper's central claim, properly identified
# =============================================================================
require("SPLITS", cell="cell 4")
# Three arms, evaluated under identical conditions:
#   chronological   the deployment-realistic reference
#   random          the naive stratified split (what v1 compared against)
#   random_matched  random, but the test block is resampled to the SAME size
#                   and the SAME positive count as the chronological test block
#
# gap_total      = random        - chronological   (what v1 reported)
# gap_prevalence = random        - random_matched  (pure class-prior artefact)
# gap_temporal   = random_matched- chronological   (the ordering effect, i.e.
#                                                  the only part attributable
#                                                  to temporal structure)
# =============================================================================

for ds in ["ULB", "PaySim"]:
    sp_chr = SPLITS[ds]
    n_te, n_pos = len(sp_chr.yte), int(sp_chr.yte.sum())
    print(f"\n=== {ds}: prevalence-controlled comparison "
          f"(target test = {n_te} rows, {n_pos} positives) ===")

    run_split(sp_chr, experiment="split_comparison", verbose=False)

    sp_rnd = random_split(ds); sp_rnd.describe()
    run_split(sp_rnd, experiment="split_comparison", verbose=False)

    for s in range(3):                       # repeat the matched arm
        sp_m = random_prevalence_matched(ds, n_te, n_pos, seed=SEED + s)
        sp_m.fold = s
        run_split(sp_m, experiment="split_comparison", verbose=False)

d = pivot("split_comparison", "PR_AUC")
tab = (d.groupby(["dataset", "split_type", "regime", "detector"])["value"]
         .mean().unstack("split_type").reset_index())
tab["gap_total"]      = tab.get("random", np.nan)         - tab["chronological"]
tab["gap_prevalence"] = tab.get("random", np.nan)         - tab.get("random_matched", np.nan)
tab["gap_temporal"]   = tab.get("random_matched", np.nan) - tab["chronological"]
tab = tab.round(4)
print("\nDecomposition of the random-vs-chronological PR-AUC gap:")
print(tab.to_string(index=False))
tab.to_csv(os.path.join(OUT_DIR, "table_split_decomposition.csv"), index=False)

share = (tab["gap_prevalence"] / tab["gap_total"].replace(0, np.nan)).median()
print(f"\n>>> Median share of the reported gap explained by the class prior "
      f"alone: {share:.1%}")
print(">>> If this is large, the v1 claim that the drop measures 'temporal "
      "leakage' does not hold and Section 4.6 must be rewritten around "
      "gap_temporal, not gap_total.")
save_results()


=== ULB: prevalence-controlled comparison (target test = 56962 rows, 75 positives) ===
  [ULB/random/fold0] train (170884, 31) (295f, 0.00173) | val (56961, 31) (98f, 0.00172) | test (56962, 31) (99f, 0.00174)
  [PaySim/random/fold0] train (3817572, 13) (4928f, 0.00129) | val (1272524, 13) (1643f, 0.00129) | test (1272524, 13) (1642f, 0.00129)

Decomposition of the random-vs-chronological PR-AUC gap:
dataset regime        detector  chronological  random  random_matched  gap_total  gap_prevalence  gap_temporal
 PaySim      N          DBSCAN         0.0248  0.0066          0.0963    -0.0182         -0.0897        0.0715
 PaySim      N        Ensemble         0.1706  0.0407          0.1134    -0.1299         -0.0727       -0.0572
 PaySim      N IsolationForest         0.0165  0.0080          0.0182    -0.0084         -0.0101        0.0017
 PaySim      N          KMeans         0.0458  0.0151          0.0362    -0.0307         -0.0211       -0.0096
 PaySim      N             LOF         0

'revision_v2\\canonical_results.csv'

In [41]:
# =============================================================================
# 7. SENSITIVITY TO rho
#    [R1#1][R6#4] label-free operation across a plausible contamination range
# =============================================================================
require("SPLITS", cell="cell 4")
# Regimes U and N are swept over the a-priori grid. Regime O (rho = the true
# training prevalence) is plotted alongside as an unreachable upper bound: the
# distance between the grid and the oracle IS the cost of not knowing rho.
# PR-AUC is threshold-free and therefore invariant to rho for every detector
# except OC-SVM (whose nu enters the optimisation); the threshold-dependent
# metrics move for all of them.

for ds in ["ULB", "PaySim"]:
    sp = SPLITS[ds]
    print(f"\n=== {ds}: rho sweep ===")
    for rho in RHO_GRID:
        run_split(sp, experiment="rho_sweep", regimes=("U", "N"),
                  rho_assumed=rho, verbose=False)
        f1 = pivot("rho_sweep", "F1", dataset=ds, rho_assumed=rho)
        print(f"  rho={rho:<7} " + " | ".join(
            f"{r.detector}({r.regime})={r.value:.3f}" for r in f1.itertuples()))

for metric in ["F1", "Precision", "Recall", "PR_AUC", "n_alerts"]:
    t = (pivot("rho_sweep", metric)
         .pivot_table(index=["dataset", "regime", "detector"],
                      columns="rho_assumed", values="value").round(4))
    t.to_csv(os.path.join(OUT_DIR, f"table_rho_sweep_{metric}.csv"))
print("\nrho sweep tables written.")
save_results()


=== ULB: rho sweep ===
  rho=0.001   IsolationForest(U)=0.000 | LOF(U)=0.176 | OneClassSVM(U)=0.000 | DBSCAN(U)=0.000 | KMeans(U)=0.000 | Ensemble(U)=0.000 | IsolationForest(N)=0.000 | LOF(N)=0.643 | OneClassSVM(N)=0.000 | DBSCAN(N)=0.000 | KMeans(N)=0.000 | Ensemble(N)=0.000
  rho=0.005   IsolationForest(U)=0.091 | LOF(U)=0.077 | OneClassSVM(U)=0.190 | DBSCAN(U)=0.000 | KMeans(U)=0.132 | Ensemble(U)=0.160 | IsolationForest(N)=0.039 | LOF(N)=0.167 | OneClassSVM(N)=0.280 | DBSCAN(N)=0.000 | KMeans(N)=0.126 | Ensemble(N)=0.313
  rho=0.01    IsolationForest(U)=0.152 | LOF(U)=0.044 | OneClassSVM(U)=0.176 | DBSCAN(U)=0.025 | KMeans(U)=0.153 | Ensemble(U)=0.160 | IsolationForest(N)=0.097 | LOF(N)=0.083 | OneClassSVM(N)=0.234 | DBSCAN(N)=0.026 | KMeans(N)=0.165 | Ensemble(N)=0.219
  rho=0.02    IsolationForest(U)=0.089 | LOF(U)=0.025 | OneClassSVM(U)=0.105 | DBSCAN(U)=0.061 | KMeans(U)=0.117 | Ensemble(U)=0.092 | IsolationForest(N)=0.081 | LOF(N)=0.044 | OneClassSVM(N)=0.121 | DBSCAN(N)=0.06

'revision_v2\\canonical_results.csv'

In [42]:
# =============================================================================
# 8. HYPERPARAMETER SENSITIVITY  --  selection on VALIDATION, per dataset
#    [R1#3][R4#7][R5#3]
# =============================================================================
require("SPLITS", cell="cell 4")
# v1 froze one configuration across two datasets of different dimension, size
# and density. Selecting per dataset is not a loosening of the protocol: it is
# the direct answer to R4#7, which asks whether LOF's collapse from 0.461 (ULB)
# to 0.006 (PaySim) is intrinsic or an artefact of k=20.

GRIDS = {
    "LOF":         [{"n_neighbors": k} for k in [5, 10, 20, 50, 100]],
    "KMeans":      [{"k": k} for k in [2, 4, 8, 16, 32]],
    # eps dominates DBSCAN; v1 fixed it at the k-distance knee and swept
    # nothing. Scales are multiples of that knee estimate.      [R5#3][R4#7]
    "DBSCAN":      ([{"eps_scale": e} for e in [0.25, 0.5, 1.0, 2.0, 4.0]]
                    + [{"min_samples": m} for m in [5, 25]]),
    "OneClassSVM": [{"gamma": g} for g in ["scale", 0.01, 0.1]],
    "IsolationForest": [{"n_estimators": n} for n in [100, 200, 400]],
}


GRID_EVAL_CAP = 200_000   # rows used for GRID SEARCH ONLY, never for headline
                          # results. Scoring 1.27M PaySim rows at k=100 for
                          # every grid point is what turns this cell into an
                          # overnight run.


def select_on_validation(sp: Split, regime: str, detector: str,
                         grid: List[dict], rho: float):
    """Choose the configuration maximising PR-AUC on the VALIDATION scores.
    The test set plays no part in selection; its PR-AUC is reported only so
    the spread across the grid can be shown."""
    Xref = reference_set(sp, regime)
    if len(sp.Xva) > GRID_EVAL_CAP or len(sp.Xte) > GRID_EVAL_CAP:
        va, te = slice(-GRID_EVAL_CAP, None), slice(-GRID_EVAL_CAP, None)
        sp = Split(sp.Xtr, sp.ytr, sp.Xva[va], sp.yva[va],
                   sp.Xte[te], sp.yte[te], sp.dataset, sp.split_type, sp.fold)
        print(f"    (grid evaluated on the last {GRID_EVAL_CAP} rows)")
    rows = []
    for cfg in grid:
        kw = dict(cfg)
        if detector == "OneClassSVM":
            kw.setdefault("nu", rho)
        sc = DETECTORS[detector](Xref, sp.Xva, sp.Xte, seed=SEED, **kw)
        val_ap = average_precision_score(sp.yva, sc.val)     # validation labels
        test_ap = average_precision_score(sp.yte, sc.test)   # reported, not used
        rows.append({"config": json.dumps(cfg), "val_PR_AUC": val_ap,
                     "test_PR_AUC": test_ap, **cfg})
    df = pd.DataFrame(rows).sort_values("val_PR_AUC", ascending=False)
    return df


sens_tables = []
for ds in ["ULB", "PaySim"]:
    sp = SPLITS[ds]
    for regime in ["U", "N"]:
        rho = rho_for(sp, regime, RHO_DEFAULT)
        for det, grid in GRIDS.items():
            print(f"  [{ds}/{regime}] sweeping {det} ({len(grid)} configs)...")
            df = select_on_validation(sp, regime, det, grid, rho)
            df.insert(0, "detector", det); df.insert(0, "regime", regime)
            df.insert(0, "dataset", ds)
            sens_tables.append(df)

sens = pd.concat(sens_tables, ignore_index=True).round(4)
sens.to_csv(os.path.join(OUT_DIR, "table_hyperparameter_sensitivity.csv"),
            index=False)

# Ties on validation PR-AUC are common and a bare argmax then lets float noise
# pick the reported configuration. The tie-break below is the one recorded in
# the PHASE 0 manifest -- simplest configuration within TIE of the best -- so
# this cell, the phase-0 cell and the manuscript name the same member.  [v2.1]
SEL_TIE = 1e-4


def _num(v, default):
    try:
        return float(v) if pd.notna(v) else default
    except (TypeError, ValueError):
        return default


def _complexity(r):
    """Lower = simpler. Used only to break ties on validation PR-AUC.
    Identical to the definition used in the phase-0 cell and recorded in the
    run manifest, so the two selections cannot diverge."""
    det = r["detector"]
    if det == "LOF":             return _num(r.get("n_neighbors"), 20)
    if det == "KMeans":          return _num(r.get("k"), 8)
    if det == "IsolationForest": return _num(r.get("n_estimators"), 200)
    if det == "OneClassSVM":
        g = r.get("gamma")
        return 0.0 if (pd.isna(g) or str(g) == "scale") else 1.0 + _num(g, 0.0)
    if det == "DBSCAN":
        e = _num(r.get("eps_scale"), 1.0); m = _num(r.get("min_samples"), 10)
        return abs(np.log(e)) + abs(m - 10) / 100.0
    return 0.0


_sel = []
for _key, _g in sens.groupby(["dataset", "regime", "detector"]):
    _g = _g.copy(); _g["complexity"] = _g.apply(_complexity, axis=1)
    _top = _g.val_PR_AUC.max()
    _tied = _g[_g.val_PR_AUC >= _top - SEL_TIE].sort_values(["complexity", "config"])
    _chosen = _tied.iloc[0].copy()
    _chosen["n_tied_on_validation"] = len(_tied)
    _chosen["tied_configs"] = " | ".join(_tied.config)
    _sel.append(_chosen)
best = pd.DataFrame(_sel).reset_index(drop=True)
spread = (sens.groupby(["dataset", "regime", "detector"])["test_PR_AUC"]
              .agg(["min", "max"]).reset_index())
spread["range"] = (spread["max"] - spread["min"]).round(4)
print("\nValidation-selected configuration per dataset:")
print(best[["dataset", "regime", "detector", "config",
            "val_PR_AUC", "test_PR_AUC"]].to_string(index=False))
print("\nTest PR-AUC spread across each grid (how much the ranking owes to "
      "hyperparameters):")
print(spread.to_string(index=False))

lof = spread[(spread.detector == "LOF")]
print("\n>>> LOF spread is the decisive number for R4#7:")
print(lof.to_string(index=False))
print(">>> If PaySim LOF recovers materially above 0.006 anywhere on the grid, "
      "the 'density instability' explanation in Section 4.7 is wrong and must "
      "be replaced by a hyperparameter-sensitivity statement.")

# repeated subsampling: how much of the ranking is subsample noise?   [R4#8]
# The default "tail" rule is deterministic, so it is overridden here -- five
# repeats of a deterministic rule would report std = 0 and answer nothing.
print("\n=== Subsampling study: BOTH selection rules ===")
print("random = uniform draw (v1 and [32]); tail = most recent contiguous")
print("block. The contrast between them is the answer to R4#8.")
_saved_rule = SUBSAMPLE_RULE
sub_rows = []
for rule in ["random", "tail"]:
    SUBSAMPLE_RULE = rule
    n_seeds = N_SUBSAMPLE_SEEDS if rule == "random" else 1   # tail is fixed
    for ds in ["ULB", "PaySim"]:
        sp = SPLITS[ds]
        for regime in ["U", "N"]:
            Xref = reference_set(sp, regime)
            rho = rho_for(sp, regime, RHO_DEFAULT)
            for det in ["LOF", "OneClassSVM", "DBSCAN"]:
                for s in range(n_seeds):
                    kw = {"nu": rho} if det == "OneClassSVM" else {}
                    sc = DETECTORS[det](Xref, sp.Xva, sp.Xte, seed=SEED + s, **kw)
                    sub_rows.append({
                        "rule": rule, "dataset": ds, "regime": regime,
                        "detector": det, "seed": s,
                        "PR_AUC": average_precision_score(sp.yte, sc.test),
                        "n_ref": sc.extra.get("n_ref", sc.extra.get("n_core")),
                    })
SUBSAMPLE_RULE = _saved_rule
sub = pd.DataFrame(sub_rows)
subs = (sub.groupby(["dataset", "regime", "detector", "rule"])["PR_AUC"]
           .agg(["mean", "std", "min", "max"]).round(4).reset_index())
print(subs.to_string(index=False))

contrast = (sub.groupby(["dataset", "regime", "detector", "rule"])["PR_AUC"]
              .mean().unstack("rule").reset_index())
contrast["rule_effect"] = (contrast["random"] - contrast["tail"]).round(4)
contrast["ratio"] = (contrast["random"] /
                     contrast["tail"].replace(0, np.nan)).round(1)
print("\n=== Effect of the selection rule alone ===")
print(contrast.round(4).to_string(index=False))
print("\n>>> Compare rule_effect to every other effect in the paper. Where it "
      "dominates, the detector's reference set matters more than its "
      "hyperparameters, its regime, or the split protocol -- and reporting a "
      "single subsample without saying which rule produced it, as v1 did, is "
      "not reproducible.")
contrast.to_csv(os.path.join(OUT_DIR, "table_subsampling_rule_effect.csv"),
                index=False)
sub.to_csv(os.path.join(OUT_DIR, "table_subsampling_raw.csv"), index=False)
subs.to_csv(os.path.join(OUT_DIR, "table_subsampling.csv"), index=False)
save_results()

  [ULB/U] sweeping LOF (5 configs)...
  [ULB/U] sweeping KMeans (5 configs)...
  [ULB/U] sweeping DBSCAN (7 configs)...
  [ULB/U] sweeping OneClassSVM (3 configs)...
  [ULB/U] sweeping IsolationForest (3 configs)...
  [ULB/N] sweeping LOF (5 configs)...
  [ULB/N] sweeping KMeans (5 configs)...
  [ULB/N] sweeping DBSCAN (7 configs)...
  [ULB/N] sweeping OneClassSVM (3 configs)...
  [ULB/N] sweeping IsolationForest (3 configs)...
  [PaySim/U] sweeping LOF (5 configs)...
    (grid evaluated on the last 200000 rows)
  [PaySim/U] sweeping KMeans (5 configs)...
    (grid evaluated on the last 200000 rows)
  [PaySim/U] sweeping DBSCAN (7 configs)...
    (grid evaluated on the last 200000 rows)
  [PaySim/U] sweeping OneClassSVM (3 configs)...
    (grid evaluated on the last 200000 rows)
  [PaySim/U] sweeping IsolationForest (3 configs)...
    (grid evaluated on the last 200000 rows)
  [PaySim/N] sweeping LOF (5 configs)...
    (grid evaluated on the last 200000 rows)
  [PaySim/N] sweeping KMea

'revision_v2\\canonical_results.csv'

In [ ]:
# =============================================================================
# 8b. THE ENSEMBLE AT VALIDATION-SELECTED CONFIGURATIONS               [v2.1]
#     v2 swept the five detectors but never the ensemble, then reported the
#     ensemble as the best PaySim configuration at the FIXED settings. Those
#     two facts sit in different tables and the comparison between them is not
#     like for like: on PaySim, tuned DBSCAN and tuned OC-SVM both overtake the
#     untuned ensemble. This cell refits every member at its validation-selected
#     configuration, rebuilds the ECDF ensemble on top, and reports it on the
#     same row scale as Table 18, so the ordering can be read at one setting.
# =============================================================================
require("SPLITS", cell="cell 4")

def ensemble_at_selected(sp: Split, regime: str, rho: float,
                         selected: pd.DataFrame) -> dict:
    Xref = reference_set(sp, regime)
    sel = selected[(selected.dataset == sp.dataset)
                   & (selected.regime == regime)]
    cals, s_va, s_te, used = {}, {}, {}, {}
    for det in DETECTORS:
        row = sel[sel.detector == det]
        cfg = json.loads(row.iloc[0].config) if len(row) else {}
        kw = dict(cfg)
        if det == "OneClassSVM":
            kw.setdefault("nu", rho)
        sc = DETECTORS[det](Xref, sp.Xva, sp.Xte, seed=SEED, **kw)
        cals[det] = ValidationCalibrator(sc.val)
        s_va[det], s_te[det] = sc.val, sc.test
        used[det] = json.dumps(cfg)
    ens_va = ecdf_rank_average(cals, s_va)
    ens_te = ecdf_rank_average(cals, s_te)
    return {"dataset": sp.dataset, "regime": regime, "detector": "Ensemble",
            "config": "; ".join(f"{k}:{v}" for k, v in used.items()),
            "val_PR_AUC": round(average_precision_score(sp.yva, ens_va), 4),
            "test_PR_AUC": round(average_precision_score(sp.yte, ens_te), 4)}


ens_sel = pd.DataFrame([ensemble_at_selected(SPLITS[ds], rg,
                                             rho_for(SPLITS[ds], rg, RHO_DEFAULT),
                                             best)
                        for ds in ["ULB", "PaySim"] for rg in ["U", "N"]])
ens_sel.to_csv(os.path.join(OUT_DIR, "table_ensemble_at_selected.csv"),
               index=False)
print("Ensemble refitted at the validation-selected member configurations:")
print(ens_sel.to_string(index=False))

print("\nSide by side with the tuned individual detectors (test PR-AUC):")
cmp = pd.concat([best[["dataset", "regime", "detector", "test_PR_AUC"]],
                 ens_sel[["dataset", "regime", "detector", "test_PR_AUC"]]],
                ignore_index=True)
for (ds, rg), g in cmp.groupby(["dataset", "regime"]):
    g = g.sort_values("test_PR_AUC", ascending=False)
    print(f"  [{ds}/{rg}] " + " > ".join(f"{r.detector} {r.test_PR_AUC:.4f}"
                                          for r in g.itertuples()))
print("\n>>> Whichever way this comes out, Sections 4.7 and 4.9 must state the "
      "ordering AT A SINGLE SETTING. Comparing a tuned detector with an untuned "
      "ensemble, as v2 did across two tables, is not a comparison.")


In [43]:
# =============================================================================
# 8bis. AGGREGATION SCHEMES  --  is rank averaging actually the right choice?
#     [R1#5] v1 proposed rank averaging without comparing it to anything, so
#     "the most appropriate aggregation strategy" was asserted, not shown.
# =============================================================================
require("SPLITS", "paired_bootstrap_diff", cell="cells 3 and 4")
# All five schemes reuse the SAME detector scores, so this costs one extra fit
# per (dataset, regime) and nothing more. Every combiner is fitted on
# validation and applied pointwise to test -- the invariants still hold.
#
#   ecdf_mean     mean of validation-ECDF transforms         (the paper's rule)
#   z_mean        mean of validation-standardised scores
#   ap_weighted   ECDF mean weighted by each detector's VALIDATION PR-AUC
#   ecdf_max      max ECDF -- an "any detector fires" rule
#   stacking      logistic regression on validation scores == the hybrid bridge,
#                 listed here to make explicit that it is SUPERVISED and hence
#                 not a competitor to the label-free combiners above

def aggregation_comparison(sp: Split, regime: str, rho: float = RHO_DEFAULT):
    Xref = reference_set(sp, regime)
    cals, v, t = {}, {}, {}
    for name, fn in DETECTORS.items():
        kw = {"nu": rho} if name == "OneClassSVM" else {}
        sc = fn(Xref, sp.Xva, sp.Xte, seed=SEED, **kw)
        cals[name] = ValidationCalibrator(sc.val)
        v[name], t[name] = sc.val, sc.test

    names = list(DETECTORS)
    Ev = np.column_stack([cals[m].ecdf(v[m]) for m in names])
    Et = np.column_stack([cals[m].ecdf(t[m]) for m in names])
    mu = np.array([v[m].mean() for m in names])
    sd = np.array([v[m].std() + 1e-12 for m in names])
    Zt = (np.column_stack([t[m] for m in names]) - mu) / sd
    w = np.array([average_precision_score(sp.yva, v[m]) for m in names])
    w = w / w.sum()

    combos = {
        "ecdf_mean":   Et.mean(axis=1),
        "z_mean":      Zt.mean(axis=1),
        "ap_weighted": Et @ w,
        "ecdf_max":    Et.max(axis=1),
    }
    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(Ev, sp.yva)
    combos["stacking_SUPERVISED"] = clf.predict_proba(Et)[:, 1]

    rows = []
    for cname, s in combos.items():
        cal = ValidationCalibrator(
            clf.predict_proba(Ev)[:, 1] if cname.startswith("stacking") else
            (Ev.mean(axis=1) if cname == "ecdf_mean" else
             ((np.column_stack([v[m] for m in names]) - mu) / sd).mean(axis=1)
             if cname == "z_mean" else
             (Ev @ w) if cname == "ap_weighted" else Ev.max(axis=1)))
        m = compute_metrics(sp.yte, cal.predict(s, rho), s)
        rows.append({"dataset": sp.dataset, "regime": regime,
                     "scheme": cname, "PR_AUC": m["PR_AUC"],
                     "F1": m["F1"], "Precision": m["Precision"],
                     "Recall": m["Recall"], "n_alerts": m["n_alerts"]})
    # best single member, as the floor any ensemble must clear
    best = max((average_precision_score(sp.yte, t[m]), m) for m in names)
    rows.append({"dataset": sp.dataset, "regime": regime,
                 "scheme": "best_single", "best_member": best[1],
                 "PR_AUC": best[0],
                 "F1": np.nan, "Precision": np.nan, "Recall": np.nan,
                 "n_alerts": np.nan})
    return pd.DataFrame(rows), {k: v_ for k, v_ in combos.items()}, sp.yte


agg_rows, AGG_SCORES = [], {}
for ds in ["ULB", "PaySim"]:
    for regime in ["U", "N"]:
        df_, sc_, y_ = aggregation_comparison(SPLITS[ds], regime)
        agg_rows.append(df_); AGG_SCORES[(ds, regime)] = (y_, sc_)
agg = pd.concat(agg_rows, ignore_index=True).round(4)
print("=== Aggregation schemes (PR-AUC on the chronological test set) ===")
print(agg.pivot_table(index=["dataset", "regime"], columns="scheme",
                      values="PR_AUC").to_string())
agg.to_csv(os.path.join(OUT_DIR, "table_aggregation_schemes.csv"), index=False)

print("\n=== Is ecdf_mean significantly better than the alternatives? ===")
for (ds, regime), (y, sc) in AGG_SCORES.items():
    for rival in ["z_mean", "ap_weighted", "ecdf_max"]:
        d, pwin, (lo, hi) = paired_bootstrap_diff(y, sc["ecdf_mean"], sc[rival],
                                                  n_boot=500)
        verdict = ("ecdf_mean ahead" if lo > 0 else
                   "ecdf_mean BEHIND" if hi < 0 else "tie")
        print(f"  [{ds}/{regime}] ecdf_mean - {rival:<12} {d:+.4f} "
              f"[{lo:+.4f}, {hi:+.4f}] -> {verdict}")

print("\n>>> If most comparisons return 'tie', the honest claim is that rank "
      "averaging is AS GOOD AS the alternatives at no tuning cost -- which is "
      "a defensible contribution. Claiming it is the BEST aggregation, as v1 "
      "did, requires at least one 'ahead' verdict.")

=== Aggregation schemes (PR-AUC on the chronological test set) ===
scheme          ap_weighted  best_single  ecdf_max  ecdf_mean  stacking_SUPERVISED  z_mean
dataset regime                                                                            
PaySim  N            0.1106       0.1018    0.0042     0.1706               0.1520  0.0085
        U            0.0900       0.1054    0.0043     0.1502               0.0862  0.0089
ULB     N            0.3440       0.6032    0.4068     0.1400               0.0653  0.1916
        U            0.0724       0.0839    0.0796     0.0695               0.0430  0.0829

=== Is ecdf_mean significantly better than the alternatives? ===
  [ULB/U] ecdf_mean - z_mean       -0.0131 [-0.0380, +0.0082] -> tie
  [ULB/U] ecdf_mean - ap_weighted  -0.0031 [-0.0192, +0.0101] -> tie
  [ULB/U] ecdf_mean - ecdf_max     -0.0129 [-0.0506, +0.0163] -> tie
  [ULB/N] ecdf_mean - z_mean       -0.0549 [-0.0949, -0.0204] -> ecdf_mean BEHIND
  [ULB/N] ecdf_mean - ap_weighte

In [44]:
# =============================================================================
# 8ter. PaySim FEATURE ABLATION
#     [R4#9] delta_orig and delta_dest are algebraic rearrangements of the
#     simulator's own account-update equations. If PaySim performance collapses
#     without them, the "cross-dataset generalisation" claim is really a claim
#     about the simulator's internals, and Sections 4.7/4.8/5 must say so.
# =============================================================================

global PAYSIM_BALANCE_FEATURES
abl_rows = []
for use_delta in [True, False]:
    PAYSIM_BALANCE_FEATURES = use_delta
    sp = chronological_split("PaySim")
    tag = "with_delta" if use_delta else "without_delta"
    print(f"\n--- PaySim {tag} ({sp.Xtr.shape[1]} features) ---")
    for regime in ["U", "N"]:
        Xref = reference_set(sp, regime)
        rho = rho_for(sp, regime, RHO_DEFAULT)
        cals, s_te = {}, {}
        for name, fn in DETECTORS.items():
            kw = {"nu": rho} if name == "OneClassSVM" else {}
            sc = fn(Xref, sp.Xva, sp.Xte, seed=SEED, **kw)
            cals[name] = ValidationCalibrator(sc.val); s_te[name] = sc.test
        s_te["Ensemble"] = ecdf_rank_average(cals, {k: s_te[k] for k in cals})
        for name, s in s_te.items():
            abl_rows.append({"features": tag, "regime": regime,
                             "detector": name,
                             "PR_AUC": average_precision_score(sp.yte, s)})
            print(f"    {regime} {name:<16} PR-AUC {abl_rows[-1]['PR_AUC']:.4f}")
PAYSIM_BALANCE_FEATURES = True          # restore the default

# --- how label-like are these features, exactly? --------------------------- #
print("\n=== The residuals used directly as anomaly scores (no model) ===")
PAYSIM_BALANCE_FEATURES = True
_sp = chronological_split("PaySim")
_X, _y, _, _, _ = load_raw("PaySim")
_cols = list(_X.columns)
probe_rows = []
i0 = int(len(_X) * (TRAIN_FRAC + VAL_FRAC))
for feat in ["delta_orig", "delta_dest"]:
    if feat not in _cols:
        continue
    raw = np.abs(_X[feat].values[i0:])
    probe_rows.append({"score": f"|{feat}| (raw, unfitted)",
                       "PR_AUC": float(average_precision_score(_y[i0:], raw)),
                       "ROC_AUC": float(roc_auc_score(_y[i0:], raw))})
both = np.abs(_X["delta_orig"].values[i0:]) + np.abs(_X["delta_dest"].values[i0:])
probe_rows.append({"score": "|delta_orig| + |delta_dest|",
                   "PR_AUC": float(average_precision_score(_y[i0:], both)),
                   "ROC_AUC": float(roc_auc_score(_y[i0:], both))})
probe = pd.DataFrame(probe_rows).round(4)
print(probe.to_string(index=False))
probe.to_csv(os.path.join(OUT_DIR, "table_paysim_residual_probe.csv"), index=False)
print(">>> Compare these to the fitted detectors above. A raw residual that "
      "out-ranks every trained model is not a feature -- it is the simulator's "
      "update rule read backwards, and PaySim cannot then be described as "
      "independent validation.")
del _X, _y, _sp

abl = (pd.DataFrame(abl_rows)
         .pivot_table(index=["regime", "detector"], columns="features",
                      values="PR_AUC").reset_index())
abl["delta_effect"] = (abl["with_delta"] - abl["without_delta"]).round(4)
abl["share_from_delta"] = (abl["delta_effect"] /
                           abl["with_delta"].replace(0, np.nan)).round(3)
print("\n=== Contribution of the simulator-derived features ===")
print(abl.round(4).to_string(index=False))
abl.to_csv(os.path.join(OUT_DIR, "table_paysim_ablation.csv"), index=False)
print("\n>>> A large share_from_delta means PaySim results are driven by "
      "features reconstructed from the simulator's update rules. Report the "
      "without_delta column as the honest cross-dataset evidence and narrow "
      "the generalisation claim accordingly.")


--- PaySim with_delta (13 features) ---
    U IsolationForest  PR-AUC 0.0161
    U LOF              PR-AUC 0.0031
    U OneClassSVM      PR-AUC 0.1054
    U DBSCAN           PR-AUC 0.0248
    U KMeans           PR-AUC 0.0419
    U Ensemble         PR-AUC 0.1502
    N IsolationForest  PR-AUC 0.0165
    N LOF              PR-AUC 0.0030
    N OneClassSVM      PR-AUC 0.1018
    N DBSCAN           PR-AUC 0.0248
    N KMeans           PR-AUC 0.0458
    N Ensemble         PR-AUC 0.1706

--- PaySim without_delta (11 features) ---
    U IsolationForest  PR-AUC 0.0143
    U LOF              PR-AUC 0.0026
    U OneClassSVM      PR-AUC 0.0552
    U DBSCAN           PR-AUC 0.0556
    U KMeans           PR-AUC 0.0369
    U Ensemble         PR-AUC 0.0191
    N IsolationForest  PR-AUC 0.0137
    N LOF              PR-AUC 0.0025
    N OneClassSVM      PR-AUC 0.0470
    N DBSCAN           PR-AUC 0.0226
    N KMeans           PR-AUC 0.0369
    N Ensemble         PR-AUC 0.0420

=== The residuals used dir

In [45]:
# =============================================================================
# 9. LABEL-EFFICIENCY CURVE  --  the hybrid bridge, honestly measured
#    [R1#6] how many labels are needed   [R4#6] v1 consumed the WHOLE
#    validation split (56,961 rows / 57 frauds), not "a few dozen labels"
#    [R6#5] no learning curve behind the "half the gap" claim
# =============================================================================
# The budget is expressed in FRAUD CASES, with the total number of labelled
# transactions reported alongside, because those two numbers are what v1
# conflated. Each budget is repeated over several draws; the spread is part of
# the result, not noise to be hidden.

# Budgets are numbers of LABELLED TRANSACTIONS drawn uniformly from the
# validation split -- an analyst reviews transactions, not frauds, so the fraud
# count is an OUTCOME of the budget, not an input. Both are reported, because
# conflating them is exactly what R4#6 and R6#5 caught.
ROW_BUDGETS = [500, 1000, 2500, 5000, 10000, 25000, None]   # None = full split
N_BUDGET_SEEDS = 20


# NOTE ON COMPARABILITY WITH v1
# v1's hybrid consumed FOUR RAW detector scores (IF, LOF, OC-SVM, K-Means;
# DBSCAN was excluded because its score was binary). v2 feeds FIVE
# validation-ECDF features, DBSCAN included, now that it emits a continuous
# score. The feature basis has changed, so the v1 headline of PR-AUC 0.544
# CANNOT be carried over -- it must be recomputed. `members` below selects the
# basis, and both are reported so the delta is visible rather than silent.

def label_efficiency(sp: Split, regime: str = "N", rho: float = RHO_DEFAULT,
                     members: Optional[List[str]] = None):
    members = members or list(DETECTORS)
    Xref = reference_set(sp, regime)
    cals, S_va, S_te = {}, {}, {}
    for name in members:
        kw = {"nu": rho} if name == "OneClassSVM" else {}
        sc = DETECTORS[name](Xref, sp.Xva, sp.Xte, seed=SEED, **kw)
        cal = ValidationCalibrator(sc.val)
        cals[name] = cal
        S_va[name] = cal.ecdf(sc.val)
        S_te[name] = cal.ecdf(sc.test)

    Mva = np.column_stack([S_va[m] for m in members])
    Mte = np.column_stack([S_te[m] for m in members])
    ens_te = Mte.mean(axis=1)
    baseline = average_precision_score(sp.yte, ens_te)

    rows = []
    budgets = [b for b in ROW_BUDGETS if b is None or b < len(sp.yva)]
    if None not in budgets:
        budgets.append(None)          # the full split is always the last point
    for budget in budgets:
        for s in range(1 if budget is None else N_BUDGET_SEEDS):
            rs = np.random.default_rng(SEED + s)
            if budget is None or budget >= len(sp.yva):
                idx = np.arange(len(sp.yva))
            else:
                idx = rs.choice(len(sp.yva), budget, replace=False)
            n_fraud = int(sp.yva[idx].sum())
            if n_fraud < 2:          # a classifier needs both classes present
                rows.append({"budget": budget or len(sp.yva),
                             "n_frauds_drawn": n_fraud, "seed": s,
                             "PR_AUC": np.nan, "degenerate": 1})
                continue
            clf = LogisticRegression(max_iter=2000, class_weight="balanced")
            clf.fit(Mva[idx], sp.yva[idx])
            p = clf.predict_proba(Mte)[:, 1]
            rows.append({"budget": budget or len(sp.yva),
                         "n_frauds_drawn": n_fraud, "seed": s,
                         "PR_AUC": float(average_precision_score(sp.yte, p)),
                         "degenerate": 0})
    df = pd.DataFrame(rows)
    out = (df.groupby("budget")
             .agg(frauds_mean=("n_frauds_drawn", "mean"),
                  degenerate=("degenerate", "sum"),
                  mean=("PR_AUC", "mean"), std=("PR_AUC", "std"),
                  n=("PR_AUC", "count"),
                  lo=("PR_AUC", lambda v: np.nanpercentile(v, 2.5)),
                  hi=("PR_AUC", lambda v: np.nanpercentile(v, 97.5)))
             .reset_index().round(4))
    out.insert(0, "members", "+".join(members))
    out.insert(0, "dataset", sp.dataset)
    out.attrs["baseline"] = baseline
    return out, baseline, df


V1_MEMBERS = ["IsolationForest", "LOF", "OneClassSVM", "KMeans"]  # v1 basis

print("=== Label-efficiency curve (ULB, regime N) ===")
le_ulb, le_base, le_raw = label_efficiency(SPLITS["ULB"])
le_v1, le_v1_base, _ = label_efficiency(SPLITS["ULB"], members=V1_MEMBERS)
print(f"  v2 basis (5 ECDF features, DBSCAN included)")
print(f"    ensemble baseline (0 labels): PR-AUC {le_base:.4f}")
print(f"    full-label hybrid           : PR-AUC {le_ulb.iloc[-1]['mean']:.4f}")
print(f"  v1 basis (4 detectors, for comparability only)")
print(f"    ensemble baseline (0 labels): PR-AUC {le_v1_base:.4f}")
print(f"    full-label hybrid           : PR-AUC {le_v1.iloc[-1]['mean']:.4f}")
print("  >>> v1 reported 0.2479 and 0.5438 under the OLD thresholding and "
      "rank rules. Neither number survives the correction; quote the values "
      "above and say in the response letter that the basis changed.")
pd.concat([le_ulb, le_v1]).to_csv(
    os.path.join(OUT_DIR, "table_label_efficiency_both_bases.csv"), index=False)
print(le_ulb.to_string(index=False))
le_ulb.to_csv(os.path.join(OUT_DIR, "table_label_efficiency.csv"), index=False)
le_raw.to_csv(os.path.join(OUT_DIR, "table_label_efficiency_raw.csv"), index=False)

full = le_ulb.iloc[-1]
print(f"\n  full split = {int(full['budget'])} labelled rows "
      f"({full['frauds_mean']:.0f} frauds): PR-AUC {full['mean']:.3f}")
for _, r in le_ulb.iloc[:-1].iterrows():
    overlap = (r["hi"] >= full["lo"]) and (full["hi"] >= r["lo"])
    print(f"  {int(r['budget']):>6} labelled rows "
          f"(~{r['frauds_mean']:.1f} frauds, {int(r['degenerate'])} draws with "
          f"<2 frauds): PR-AUC {r['mean']:.3f} +/- {r['std']:.3f}  -> "
          f"{'INDISTINGUISHABLE from' if overlap else 'BELOW'} the full-label point")

print("\n>>> Report BOTH columns in Section 4.5. The v1 phrase 'a few dozen "
      "labeled fraud cases' described a run that consumed the entire "
      "validation partition; the budgets above are what that phrase promised.")

=== Label-efficiency curve (ULB, regime N) ===
  v2 basis (5 ECDF features, DBSCAN included)
    ensemble baseline (0 labels): PR-AUC 0.1400
    full-label hybrid           : PR-AUC 0.0653
  v1 basis (4 detectors, for comparability only)
    ensemble baseline (0 labels): PR-AUC 0.1626
    full-label hybrid           : PR-AUC 0.1220
  >>> v1 reported 0.2479 and 0.5438 under the OLD thresholding and rank rules. Neither number survives the correction; quote the values above and say in the response letter that the basis changed.
dataset                                       members  budget  frauds_mean  degenerate   mean    std  n     lo     hi
    ULB IsolationForest+LOF+OneClassSVM+DBSCAN+KMeans     500         0.60          16 0.1201 0.0791  4 0.0132 0.1634
    ULB IsolationForest+LOF+OneClassSVM+DBSCAN+KMeans    1000         0.95          16 0.1771 0.0187  4 0.1607 0.1958
    ULB IsolationForest+LOF+OneClassSVM+DBSCAN+KMeans    2500         2.45           5 0.1477 0.0587 15 0.0058 0.19

In [ ]:
# =============================================================================
# 10. UNCERTAINTY  --  block bootstrap over the chronological test stream
#     [R5#2] confidence intervals   [R6#3] 75 frauds is not enough for
#     fine-grained rankings, and the intervals should say so
# =============================================================================
# The helpers live in the calibration cell so that every experiment can use
# them; this cell applies them to the headline configurations.

print("=== Block bootstrap CIs (chronological test sets) ===")
ci_rows, SCORE_CACHE = [], {}
for ds in ["ULB", "PaySim"]:
    sp = SPLITS[ds]
    for regime in ["U", "N"]:
        Xref = reference_set(sp, regime)
        rho = rho_for(sp, regime, RHO_DEFAULT)
        cals, s_te = {}, {}
        for name, fn in DETECTORS.items():
            kw = {"nu": rho} if name == "OneClassSVM" else {}
            sc = fn(Xref, sp.Xva, sp.Xte, seed=SEED, **kw)
            cals[name] = ValidationCalibrator(sc.val)
            s_te[name] = sc.test
        s_te["Ensemble"] = ecdf_rank_average(cals, {k: s_te[k] for k in cals})
        SCORE_CACHE[(ds, regime)] = (sp.yte, s_te)
        for name, s in s_te.items():
            (lo, med, hi), _ = block_bootstrap_ap(sp.yte, s)
            ci_rows.append({"dataset": ds, "regime": regime, "detector": name,
                            "PR_AUC": average_precision_score(sp.yte, s),
                            "ci_lo": lo, "ci_med": med, "ci_hi": hi,
                            "ci_width": hi - lo})
            print(f"  [{ds}/{regime}] {name:<16} "
                  f"{average_precision_score(sp.yte, s):.4f}  "
                  f"[{lo:.4f}, {hi:.4f}]")

ci = pd.DataFrame(ci_rows).round(4)
ci.to_csv(os.path.join(OUT_DIR, "table_bootstrap_ci.csv"), index=False)

# save the raw scores so this cell never fails for want of a file again
np.savez_compressed(
    os.path.join(OUT_DIR, "test_scores.npz"),
    **{f"{ds}|{rg}|{det}": s
       for (ds, rg), (_, d) in SCORE_CACHE.items() for det, s in d.items()},
    **{f"{ds}|{rg}|__y": y for (ds, rg), (y, _) in SCORE_CACHE.items()})

print("\n=== Paired comparison: is the ensemble really ahead? ===")
for (ds, regime), (y, s) in SCORE_CACHE.items():
    rivals = sorted(((average_precision_score(y, v), k)
                     for k, v in s.items() if k != "Ensemble"), reverse=True)
    best_ap, best_name = rivals[0]
    d, pwin, (lo, hi) = paired_bootstrap_diff(y, s["Ensemble"], s[best_name])
    verdict = ("ensemble ahead" if lo > 0 else
               "ENSEMBLE BEHIND" if hi < 0 else "WITHIN NOISE")
    print(f"  [{ds}/{regime}] Ensemble - {best_name}: {d:+.4f} "
          f"[{lo:+.4f}, {hi:+.4f}]  P(win)={pwin:.2f}  -> {verdict}")
print("\n>>> WITHIN NOISE  -> the claim loses its causal verb in the "
      "manuscript: 'confirms'/'establishes'/'demonstrates' become 'is "
      "consistent with'.")
print(">>> ENSEMBLE BEHIND -> Section 4.4's 'the ensemble does not depend on "
      "any single member' still stands, but the ranking claim does not. Say so.")

=== Block bootstrap CIs (chronological test sets) ===
  [ULB/U] IsolationForest  0.0546  [0.0314, 0.0910]
  [ULB/U] LOF              0.0763  [0.0301, 0.1594]
  [ULB/U] OneClassSVM      0.0839  [0.0517, 0.1304]
  [ULB/U] DBSCAN           0.0247  [0.0160, 0.0375]
  [ULB/U] KMeans           0.0679  [0.0418, 0.1043]
  [ULB/U] Ensemble         0.0695  [0.0433, 0.1068]
  [ULB/N] IsolationForest  0.0369  [0.0221, 0.0607]
  [ULB/N] LOF              0.6032  [0.4458, 0.7512]
  [ULB/N] OneClassSVM      0.1248  [0.0809, 0.1837]
  [ULB/N] DBSCAN           0.0250  [0.0163, 0.0378]
  [ULB/N] KMeans           0.0673  [0.0409, 0.1062]
  [ULB/N] Ensemble         0.1400  [0.0883, 0.2112]
  [PaySim/U] IsolationForest  0.0161  [0.0089, 0.0246]
  [PaySim/U] LOF              0.0031  [0.0017, 0.0045]
  [PaySim/U] KMeans           0.0419  [0.0217, 0.0673]
  [PaySim/N] OneClassSVM      0.1018  [0.0527, 0.1637]


In [50]:
# =============================================================================
# 11. OPERATIONAL MEASUREMENT  --  inference, not fitting
#     [R4#10] v1's Section 4.8 timed model FITTING and then claimed deployment
#     readiness   [R5#4] scalability in a real financial system
# =============================================================================
require("SCORE_CACHE", "SPLITS", cell="cell 13")
# Once thresholds and ECDFs are frozen on validation, the detectors genuinely
# become streaming-compatible -- which v1's batch ranking was not. That is the
# argument to make in the response letter: the correction does not weaken the
# operational claim, it is what licenses it.

import tracemalloc


def streaming_profile(sp: Split, regime: str = "N", rho: float = RHO_DEFAULT,
                      n_probe: int = 2000):
    Xref = reference_set(sp, regime)
    rows = []
    rs = np.random.default_rng(SEED)
    probe = rs.choice(len(sp.Xte), min(n_probe, len(sp.Xte)), replace=False)
    keep = {}                    # kept so the ensemble can be profiled too
    for name, fn in DETECTORS.items():
        kw = {"nu": rho} if name == "OneClassSVM" else {}
        tracemalloc.start()
        sc = fn(Xref, sp.Xva, sp.Xte, seed=SEED, **kw)
        _, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
        cal = ValidationCalibrator(sc.val)
        tau = cal.threshold(rho)
        keep[name] = (sc, cal, peak)

        lat = []
        for i in probe[:200]:            # score ONE transaction, end to end
            z = sp.Xte[i:i + 1]
            t0 = time.perf_counter()
            _ = sc.score_one(z) >= tau   # full scoring + decision, no batch
            lat.append((time.perf_counter() - t0) * 1e6)   # microseconds
        n_alerts = int((sc.test >= tau).sum())
        rows.append({
            "dataset": sp.dataset, "regime": regime, "detector": name,
            "fit_s": round(sc.fit_seconds, 2),
            "score_s_per_1k": round(sc.score_seconds / len(sp.Xte) * 1000, 4),
            "decision_us_p50": round(float(np.percentile(lat, 50)), 3),
            "decision_us_p95": round(float(np.percentile(lat, 95)), 3),
            "peak_mem_MB": round(peak / 1e6, 1),
            "alerts": n_alerts,
            "alerts_per_10k": round(n_alerts / len(sp.Xte) * 10_000, 1),
        })

    # The ensemble has no fit or score of its own: it consumes the five member
    # scores and adds five ECDF lookups. v2 omitted it from this table while
    # naming it the best PaySim configuration, so its cost was unreported. It
    # is profiled here by summing the members and timing the lookups.   [v2.1]
    ens_val = ecdf_rank_average({k: v[1] for k, v in keep.items()},
                                {k: v[0].val for k, v in keep.items()})
    ens_test = ecdf_rank_average({k: v[1] for k, v in keep.items()},
                                 {k: v[0].test for k, v in keep.items()})
    ens_cal = ValidationCalibrator(ens_val)
    ens_tau = ens_cal.threshold(rho)
    lat = []
    for i in probe[:200]:
        z = sp.Xte[i:i + 1]
        t0 = time.perf_counter()
        _ = (np.mean([keep[k][1].ecdf(np.array([keep[k][0].score_one(z)]))[0]
                      for k in keep]) >= ens_tau)
        lat.append((time.perf_counter() - t0) * 1e6)
    n_alerts = int((ens_test >= ens_tau).sum())
    rows.append({
        "dataset": sp.dataset, "regime": regime, "detector": "Ensemble",
        "fit_s": round(sum(v[0].fit_seconds for v in keep.values()), 2),
        "score_s_per_1k": round(sum(v[0].score_seconds for v in keep.values())
                                / len(sp.Xte) * 1000, 4),
        "decision_us_p50": round(float(np.percentile(lat, 50)), 3),
        "decision_us_p95": round(float(np.percentile(lat, 95)), 3),
        "peak_mem_MB": round(sum(v[2] for v in keep.values()) / 1e6, 1),
        "alerts": n_alerts,
        "alerts_per_10k": round(n_alerts / len(sp.Xte) * 10_000, 1),
    })
    return pd.DataFrame(rows)


ops = pd.concat([streaming_profile(SPLITS[ds]) for ds in ["ULB", "PaySim"]],
                ignore_index=True)


def scaling_curve(sp: Split, regime: str = "N", rho: float = RHO_DEFAULT,
                  sizes=(10_000, 50_000, 100_000, 200_000)):
    """Fit cost as a function of training-set size -- the concrete form of
    'scalability in real-world financial systems'.                     [R5#4]"""
    Xref_full = reference_set(sp, regime)
    rows = []
    for n in sizes:
        if n > len(Xref_full):
            continue
        Xref = Xref_full[-n:]                       # same tail rule as elsewhere
        for name, fn in DETECTORS.items():
            kw = {"nu": rho} if name == "OneClassSVM" else {}
            sc = fn(Xref, sp.Xva[:5000], sp.Xte[:5000], seed=SEED, **kw)
            rows.append({"dataset": sp.dataset, "detector": name, "n_train": n,
                         "fit_s": round(sc.fit_seconds, 3),
                         "score_s_per_1k": round(sc.score_seconds / 5.0, 4)})
    return pd.DataFrame(rows)


scal = pd.concat([scaling_curve(SPLITS[ds]) for ds in ["ULB", "PaySim"]],
                 ignore_index=True)
print("\n=== Fit cost vs training-set size ===")
print(scal.pivot_table(index=["dataset", "detector"], columns="n_train",
                       values="fit_s").to_string())
scal.to_csv(os.path.join(OUT_DIR, "table_scaling.csv"), index=False)
print(ops.to_string(index=False))
ops.to_csv(os.path.join(OUT_DIR, "table_operational.csv"), index=False)

# Alert budget: recall achievable if analysts can review K alerts per 10k tx
print("\n=== Recall under a fixed analyst capacity ===")
cap_rows = []
for (ds, regime), (y, s) in SCORE_CACHE.items():
    for per10k in [1, 5, 10, 50]:
        k = max(1, int(len(y) * per10k / 10_000))
        for det, sc in s.items():
            top = np.argpartition(-sc, k)[:k]
            cap_rows.append({"dataset": ds, "regime": regime, "detector": det,
                             "alerts_per_10k": per10k, "k": k,
                             "recall": round(float(y[top].sum() / max(y.sum(), 1)), 4),
                             "precision": round(float(y[top].mean()), 4)})
cap = pd.DataFrame(cap_rows)
print(cap[cap.alerts_per_10k == 10].to_string(index=False))
cap.to_csv(os.path.join(OUT_DIR, "table_alert_budget.csv"), index=False)


=== Fit cost vs training-set size ===
n_train                  10000   50000   100000  200000
dataset detector                                       
PaySim  DBSCAN            0.415   1.481   0.832   1.290
        IsolationForest   0.569   0.946   1.083   1.324
        KMeans            0.372   1.304   2.365   4.554
        LOF               0.477   1.744   0.999   2.068
        OneClassSVM       0.063   0.251   0.242   0.229
ULB     DBSCAN            0.191   0.706   0.698     NaN
        IsolationForest   0.608   0.973   1.107     NaN
        KMeans            0.560   1.932   3.793     NaN
        LOF               0.313   0.812   0.824     NaN
        OneClassSVM       0.270   0.826   0.821     NaN
dataset regime        detector  fit_s  score_s_per_1k  decision_us_p50  decision_us_p95  peak_mem_MB  alerts  alerts_per_10k
    ULB      N IsolationForest   6.30          0.0311          6311.60         6546.175         10.3     184            32.3
    ULB      N             LOF   1.02  

In [51]:
# =============================================================================
# 12. TABLES AND FIGURES  --  all derived from the canonical frame
#     [R4#5] the v1 figures showed the STRATIFIED run under a caption claiming
#     the chronological one. Here nothing is drawn from a variable: every plot
#     reads the frame, and an assertion compares what is drawn to what is
#     tabulated before the file is written.
# =============================================================================
require("SCORE_CACHE", "le_ulb", "le_base", cell="cells 12 and 13")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve

FIG_DIR = os.path.join(OUT_DIR, "figures"); os.makedirs(FIG_DIR, exist_ok=True)
CANON = results_frame()


def table_main(dataset: str) -> pd.DataFrame:
    d = CANON[(CANON.experiment == "headline") & (CANON.dataset == dataset)
              & (CANON.metric.isin(["Precision", "Recall", "F1", "ROC_AUC",
                                    "PR_AUC", "TP", "FP", "n_alerts"]))]
    t = d.pivot_table(index=["regime", "detector"], columns="metric",
                      values="value")
    return t[["Precision", "Recall", "F1", "ROC_AUC", "PR_AUC",
              "TP", "FP", "n_alerts"]].round(4)


def plot_pr(dataset: str, regime: str):
    """Draws from SCORE_CACHE, then ASSERTS every legend value against the
    canonical frame. A caption can no longer disagree with a table."""
    y, s = SCORE_CACHE[(dataset, regime)]
    fig, ax = plt.subplots(figsize=(7, 5))
    drawn = {}
    for name, sc in s.items():
        ap = average_precision_score(y, sc)
        drawn[name] = ap
        p, r, _ = precision_recall_curve(y, sc)
        ax.plot(r, p, lw=1.4, label=f"{name} AP={ap:.3f}")
    base = float(y.mean())
    ax.axhline(base, ls="--", c="grey", lw=1, label=f"baseline={base:.4f}")

    # ---- INVARIANT 3 ----------------------------------------------------- #
    tab = CANON[(CANON.experiment == "headline") & (CANON.dataset == dataset)
                & (CANON.regime == regime) & (CANON.metric == "PR_AUC")]
    lookup = dict(zip(tab.detector, tab.value))
    for name, ap in drawn.items():
        ref = lookup.get(name)
        assert ref is not None and abs(ref - ap) < 1e-6, (
            f"FIGURE/TABLE MISMATCH {dataset}/{regime}/{name}: "
            f"figure {ap:.6f} vs table {ref}")
    n_pos = int(y.sum())
    assert abs(base - n_pos / len(y)) < 1e-12
    # ---------------------------------------------------------------------- #

    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title(f"Precision-recall, {dataset} chronological test set "
                 f"(regime {regime}; {n_pos} frauds / {len(y)} tx, "
                 f"prevalence {base:.4f})")
    ax.legend(fontsize=7); fig.tight_layout()
    p = os.path.join(FIG_DIR, f"pr_{dataset}_{regime}.png")
    fig.savefig(p, dpi=180); plt.close(fig)
    return p


def plot_label_efficiency(le: pd.DataFrame, baseline: float, dataset="ULB"):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    x = le["budget"].values
    ax.errorbar(x, le["mean"], yerr=le["std"], marker="o", capsize=3,
                label="hybrid meta-classifier")
    ax.axhline(baseline, ls="--", c="grey",
               label=f"unsupervised-score ensemble ({baseline:.3f}, 0 labels)")
    ax.fill_between(x, le["lo"], le["hi"], alpha=0.15)
    ax.set_xscale("log")
    ax.set_xlabel("labelled transactions in the budget (log scale)")
    ax.set_ylabel("Test PR-AUC")
    ax.set_title(f"Label efficiency of the hybrid bridge ({dataset}, "
                 f"chronological test set)")
    ax.legend(fontsize=8); fig.tight_layout()
    p = os.path.join(FIG_DIR, f"label_efficiency_{dataset}.png")
    fig.savefig(p, dpi=180); plt.close(fig)
    return p


def plot_roc(dataset: str, regime: str):
    """Figure 3's counterpart. v1's ROC plot carried the same stratified-run
    values under a chronological caption, so it gets the same assertion. [R4#5]"""
    y, s = SCORE_CACHE[(dataset, regime)]
    fig, ax = plt.subplots(figsize=(7, 5))
    drawn = {}
    for name, sc in s.items():
        auc = roc_auc_score(y, sc); drawn[name] = auc
        fpr, tpr, _ = roc_curve(y, sc)
        ax.plot(fpr, tpr, lw=1.4, label=f"{name} AUC={auc:.3f}")
    ax.plot([0, 1], [0, 1], ls="--", c="grey", lw=1)

    tab = CANON[(CANON.experiment == "headline") & (CANON.dataset == dataset)
                & (CANON.regime == regime) & (CANON.metric == "ROC_AUC")]
    lookup = dict(zip(tab.detector, tab.value))
    for name, auc in drawn.items():
        ref = lookup.get(name)
        assert ref is not None and abs(ref - auc) < 1e-6, (
            f"FIGURE/TABLE MISMATCH (ROC) {dataset}/{regime}/{name}: "
            f"figure {auc:.6f} vs table {ref}")

    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_title(f"ROC, {dataset} chronological test set (regime {regime}) "
                 f"-- secondary metric, see PR curves for operational reading")
    ax.legend(fontsize=7); fig.tight_layout()
    p = os.path.join(FIG_DIR, f"roc_{dataset}_{regime}.png")
    fig.savefig(p, dpi=180); plt.close(fig)
    return p


def plot_rolling(dataset: str, regime: str = "N"):
    d = CANON[(CANON.experiment == "rolling_origin") & (CANON.dataset == dataset)
              & (CANON.regime == regime) & (CANON.metric == "PR_AUC")]
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for det, g in d.groupby("detector"):
        g = g.sort_values("fold")
        ax.plot(g.fold, g.value, marker="o", lw=1.3, label=det)
    ax.set_xlabel("rolling origin (later = further into the stream)")
    ax.set_ylabel("Test PR-AUC")
    ax.set_title(f"Stability across temporal origins ({dataset}, regime {regime})")
    ax.legend(fontsize=7); fig.tight_layout()
    p = os.path.join(FIG_DIR, f"rolling_{dataset}_{regime}.png")
    fig.savefig(p, dpi=180); plt.close(fig)
    return p


written = []
for ds in ["ULB", "PaySim"]:
    t = table_main(ds)
    print(f"\n=== Main results, {ds} (chronological) ===")
    print(t.to_string())
    t.to_csv(os.path.join(OUT_DIR, f"table_main_{ds}.csv"))
    for regime in ["U", "N"]:
        written.append(plot_pr(ds, regime))
        written.append(plot_roc(ds, regime))
    written.append(plot_rolling(ds))
written.append(plot_label_efficiency(le_ulb, le_base))

print("\nFigures written (all legend values asserted against the canonical "
      "frame):")
for p in written:
    print("  ", p)

save_results()
print("\n" + "=" * 70)
print("Traceability: every number above carries a run_id in "
      f"{OUT_DIR}/canonical_results.csv")
print("Before writing any sentence into the manuscript, look the value up "
      "there rather than copying it from a printout.")
print("=" * 70)


=== Main results, ULB (chronological) ===
metric                  Precision  Recall      F1  ROC_AUC  PR_AUC    TP     FP  n_alerts
regime detector                                                                          
N      DBSCAN              0.0000  0.0000  0.0000   0.9437  0.0250   0.0  179.0     179.0
       Ensemble            0.2176  0.5600  0.3134   0.9639  0.1400  42.0  151.0     193.0
       IsolationForest     0.0272  0.0667  0.0386   0.9442  0.0369   5.0  179.0     184.0
       KMeans              0.0894  0.2133  0.1260   0.9671  0.0673  16.0  163.0     179.0
       LOF                 0.0932  0.7867  0.1667   0.9266  0.6032  59.0  574.0     633.0
       OneClassSVM         0.1939  0.5067  0.2804   0.9539  0.1248  38.0  158.0     196.0
O      DBSCAN              0.0000  0.0000  0.0000   0.9382  0.0247   0.0   75.0      75.0
       Ensemble            0.0000  0.0000  0.0000   0.9585  0.0695   0.0   73.0      73.0
       IsolationForest     0.0000  0.0000  0.0000   0.948

In [49]:
# =============================================================================
# CELL A  --  MISSING FIGURES, rebuilt from the canonical frame only
# =============================================================================
# Paste at the end of the running notebook. Nothing is refitted: the confusion
# counts were already recorded by compute_metrics(), so these figures are drawn
# from the same rows the tables are drawn from. That makes figure/table
# agreement true by construction rather than by assertion.             [R4#5]
#
# Self-contained: if the kernel was restarted, it reloads from disk.
# =============================================================================
import os, json
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR = globals().get("OUT_DIR", "revision_v2")
FIG_DIR = globals().get("FIG_DIR", os.path.join(OUT_DIR, "figures"))
os.makedirs(FIG_DIR, exist_ok=True)

if "CANON" in globals() and len(globals()["CANON"]):
    CANON = globals()["CANON"]
else:
    CANON = pd.read_csv(os.path.join(OUT_DIR, "canonical_results.csv"))
print(f"canonical frame: {len(CANON)} rows, "
      f"{CANON.experiment.nunique()} experiments")


def _counts(dataset, regime, experiment="headline"):
    d = CANON[(CANON.experiment == experiment) & (CANON.dataset == dataset)
              & (CANON.regime == regime)
              & (CANON.metric.isin(["TN", "FP", "FN", "TP"]))]
    return d.pivot_table(index="detector", columns="metric", values="value")


def plot_confusion_grid(dataset: str, regime: str):
    """v1's Figures 4 and 9 put three detectors thresholded by scikit-learn's
    native predict() next to two thresholded by Equation 13, then compared the
    alert volumes across panels. Those panels were not comparable, and Section
    4.3 drew an operational conclusion from the comparison. Here every panel
    carries the same validation-calibrated rule, and the alert count is printed
    in each title so the comparison is explicit rather than implied."""
    cm = _counts(dataset, regime)
    if cm.empty:
        print(f"  ! no headline rows for {dataset}/{regime}")
        return None
    order = ["IsolationForest", "LOF", "OneClassSVM", "DBSCAN", "KMeans", "Ensemble"]
    names = [n for n in order if n in cm.index] + \
            [n for n in cm.index if n not in order]
    ncol = 3; nrow = int(np.ceil(len(names) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 3.0 * nrow))
    axes = np.atleast_1d(axes).ravel()

    for ax, name in zip(axes, names):
        r = cm.loc[name]
        tn, fp, fn, tp = int(r.TN), int(r.FP), int(r.FN), int(r.TP)
        M = np.array([[tn, fp], [fn, tp]])
        ax.imshow(np.log1p(M), cmap="Blues")
        hi = np.log1p(M).max()
        for (i, j), v in np.ndenumerate(M):
            ax.text(j, i, f"{v:,}", ha="center", va="center", fontsize=9,
                    color="white" if np.log1p(v) > 0.6 * hi else "black")
        ax.set_xticks([0, 1]); ax.set_xticklabels(["pred 0", "pred 1"], fontsize=8)
        ax.set_yticks([0, 1]); ax.set_yticklabels(["true 0", "true 1"], fontsize=8)
        rec = tp / max(tp + fn, 1)
        ax.set_title(f"{name}\n{tp + fp:,} alerts, recall {rec:.2f}", fontsize=9)
    for ax in axes[len(names):]:
        ax.axis("off")

    n_alerts = (cm.TP + cm.FP).astype(int)
    fig.suptitle(f"Confusion matrices, {dataset} chronological test set "
                 f"(regime {regime}). Every panel uses the same "
                 f"validation-calibrated threshold; alert volumes range "
                 f"{n_alerts.min():,}-{n_alerts.max():,}.", fontsize=10)
    fig.tight_layout()
    p = os.path.join(FIG_DIR, f"confusion_{dataset}_{regime}.png")
    fig.savefig(p, dpi=180); plt.close(fig)
    return p


def plot_split_gap(dataset: str, regime: str = "N"):
    """v1's Figure 6 showed two bars, random and chronological, and read the
    difference as temporal leakage. The two arms also differ in test prevalence,
    so that reading was not identified. The third bar holds prevalence fixed and
    the lower panel separates the two components.                       [R4#4]"""
    d = CANON[(CANON.experiment == "split_comparison") & (CANON.dataset == dataset)
              & (CANON.regime == regime) & (CANON.metric == "PR_AUC")]
    if d.empty:
        print(f"  ! no split_comparison rows for {dataset}/{regime}")
        return None
    piv = d.pivot_table(index="detector", columns="split_type",
                        values="value", aggfunc="mean")
    need = ["chronological", "random", "random_matched"]
    if any(c not in piv for c in need):
        print(f"  ! missing arms for {dataset}/{regime}: "
              f"{[c for c in need if c not in piv]}")
        return None
    piv = piv.sort_values("chronological", ascending=False)

    x = np.arange(len(piv)); w = 0.27
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9.5, 7.5),
                                   gridspec_kw={"height_ratios": [2, 1]})
    ax1.bar(x - w, piv["random"], w, label="random split")
    ax1.bar(x, piv["random_matched"], w, label="random, test prevalence matched")
    ax1.bar(x + w, piv["chronological"], w, label="chronological split")
    ax1.set_xticks(x); ax1.set_xticklabels(piv.index, rotation=18, ha="right",
                                           fontsize=9)
    ax1.set_ylabel("Test PR-AUC")
    ax1.set_title(f"Random versus chronological evaluation, {dataset} "
                  f"(regime {regime})")
    ax1.legend(fontsize=8)

    gp = piv["random"] - piv["random_matched"]
    gt = piv["random_matched"] - piv["chronological"]
    ax2.bar(x - w / 2, gp, w, label="attributable to test prevalence")
    ax2.bar(x + w / 2, gt, w, label="attributable to temporal ordering")
    ax2.axhline(0, c="grey", lw=0.8)
    ax2.set_xticks(x); ax2.set_xticklabels(piv.index, rotation=18, ha="right",
                                           fontsize=9)
    ax2.set_ylabel("PR-AUC difference")
    ax2.set_title("Decomposition of the gap; only the second component can be "
                  "read as a temporal effect", fontsize=10)
    ax2.legend(fontsize=8)
    fig.tight_layout()
    p = os.path.join(FIG_DIR, f"split_gap_{dataset}_{regime}.png")
    fig.savefig(p, dpi=180); plt.close(fig)

    out = pd.DataFrame({"random": piv["random"],
                        "random_matched": piv["random_matched"],
                        "chronological": piv["chronological"],
                        "gap_prevalence": gp, "gap_temporal": gt}).round(4)
    out.to_csv(os.path.join(OUT_DIR, f"table_split_gap_{dataset}_{regime}.csv"))
    print(f"\n  {dataset}/{regime} decomposition:")
    print(out.to_string())
    return p


written = []
for ds in sorted(CANON.dataset.unique()):
    for rg in ["U", "N"]:
        p = plot_confusion_grid(ds, rg)
        if p: written.append(p)
    p = plot_split_gap(ds, "N")
    if p: written.append(p)

print("\nFigures written:")
for p in written:
    print("  ", p)


# =============================================================================
# CELL B  --  MANUSCRIPT-READY TABLES
# =============================================================================
# Emits every table in the layout it should have in the paper, as CSV and as
# tab-separated text that pastes directly into Word (Insert > Table > Convert
# Text to Table). Read-only with respect to the results: nothing is recomputed.
# =============================================================================
TAB_DIR = os.path.join(OUT_DIR, "manuscript_tables")
os.makedirs(TAB_DIR, exist_ok=True)
REGIME_LABEL = {"U": "U (label-free)", "N": "N (normal-only)",
                "O": "O (oracle, upper bound)"}


def emit(df, name, note=""):
    df = df.copy()
    df.to_csv(os.path.join(TAB_DIR, f"{name}.csv"), index=False)
    with open(os.path.join(TAB_DIR, f"{name}.txt"), "w", encoding="utf-8") as f:
        if note:
            f.write(note + "\n\n")
        f.write(df.to_csv(sep="\t", index=False))
    print(f"\n===== {name} =====")
    if note:
        print(note)
    print(df.to_string(index=False))


# ---- Table 4 / 6 : main results per dataset ------------------------------- #
for ds in sorted(CANON.dataset.unique()):
    d = CANON[(CANON.experiment == "headline") & (CANON.dataset == ds)
              & (CANON.metric.isin(["Precision", "Recall", "F1", "ROC_AUC",
                                    "PR_AUC", "TP", "FP", "n_alerts"]))]
    t = d.pivot_table(index=["regime", "detector"], columns="metric",
                      values="value").reset_index()
    t["regime"] = t["regime"].map(REGIME_LABEL).fillna(t["regime"])
    cols = ["regime", "detector", "Precision", "Recall", "F1", "ROC_AUC",
            "PR_AUC", "TP", "FP", "n_alerts"]
    t = t[[c for c in cols if c in t]]
    for c in ["Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"]:
        if c in t: t[c] = t[c].round(4)
    for c in ["TP", "FP", "n_alerts"]:
        if c in t: t[c] = t[c].astype(int)
    emit(t, f"table_main_{ds}",
         f"{ds}, chronological test set. All configurations share one "
         f"decision rule: the (1-rho) quantile of validation scores, rho="
         f"{CANON[CANON.experiment=='headline'].rho_assumed.mode().iat[0]}. "
         f"PR-AUC is the primary metric; ROC-AUC is reported only as a "
         f"complementary measure of ranking.")

# ---- Table 7 : bootstrap intervals ---------------------------------------- #
p = os.path.join(OUT_DIR, "table_bootstrap_ci.csv")
if os.path.exists(p):
    ci = pd.read_csv(p)
    ci["PR-AUC [95% CI]"] = ci.apply(
        lambda r: f"{r.PR_AUC:.4f} [{r.ci_lo:.4f}, {r.ci_hi:.4f}]", axis=1)
    ci["regime"] = ci["regime"].map(REGIME_LABEL).fillna(ci["regime"])
    emit(ci[["dataset", "regime", "detector", "PR-AUC [95% CI]", "ci_width"]],
         "table_bootstrap_ci_formatted",
         "Block bootstrap over temporal blocks of the chronological test set, "
         "2000 replicates. Overlapping intervals do not support a ranking.")

# ---- Table 8 : rolling origins -------------------------------------------- #
p = os.path.join(OUT_DIR, "table_rolling_origin.csv")
if os.path.exists(p):
    ro = pd.read_csv(p)
    ro = ro[ro.regime.isin(["U", "N"])].copy()
    ro["regime"] = ro["regime"].map(REGIME_LABEL)
    keep = [c for c in ["dataset", "regime", "detector", "pr_mean", "pr_std",
                        "lift_mean", "cv", "n"] if c in ro]
    emit(ro[keep].round(4), "table_rolling_origin_formatted",
         "PR-AUC across rolling origins. Fold prevalence varies, so lift over "
         "each fold's own random baseline is reported alongside the raw value. "
         "cv above 0.5 indicates that the detector ordering is not stable.")

# ---- Table 13 : subsampling rule ------------------------------------------ #
p = os.path.join(OUT_DIR, "table_subsampling_rule_effect.csv")
if os.path.exists(p):
    su = pd.read_csv(p)
    su["regime"] = su["regime"].map(REGIME_LABEL).fillna(su["regime"])
    emit(su.round(4), "table_subsampling_rule_effect_formatted",
         "Effect of the reference-set selection rule alone, at fixed "
         "hyperparameters, regime and split. Compare the magnitude of "
         "rule_effect with every other effect reported in the paper.")

# ---- Table A : delta versus references [31] and [32] ---------------------- #
delta = pd.DataFrame([
    ["Detectors compared", "OC-SVM, LOF, VAE, GAN (4)",
     "supervised DNN, LR, RF, XGBoost", "IF, LOF, OC-SVM, DBSCAN, K-Means (5)"],
    ["Learning setting", "unsupervised", "fully supervised",
     "label-free, normal-only and oracle regimes"],
    ["Split protocol", "not chronological", "chronological 60/20/20",
     "chronological, plus rolling origins and a prevalence-matched random arm"],
    ["Primary metric", "ROC-AUC", "PR-AUC and expected cost", "PR-AUC"],
    ["Decision threshold", "not calibrated", "validation, cost-optimal",
     "validation quantile, applied unchanged"],
    ["Ensemble", "no", "no", "four aggregation schemes compared"],
    ["Hybrid bridge", "no", "not applicable", "label-efficiency curve"],
    ["Uncertainty", "no", "bootstrap CI on PR-AUC",
     "block bootstrap, paired tests, repeated subsampling"],
    ["Datasets", "one", "ULB", "ULB and PaySim, with a feature ablation"],
], columns=["Dimension", "Reference [31]", "Reference [32]", "This work"])
emit(delta, "table_A_delta_vs_prior_work",
     "Explicit differences with the two closest prior studies by the same "
     "author group. VERIFY the [31] column against the published paper before "
     "submission; it is filled from the abstract and must not be guessed.")

print(f"\n\nAll manuscript tables written to {os.path.abspath(TAB_DIR)}")
print("Each has a .csv and a .txt; the .txt is tab-separated and pastes into "
      "Word via Insert > Table > Convert Text to Table.")

canonical frame: 7372 rows, 4 experiments

  PaySim/N decomposition:
                 random  random_matched  chronological  gap_prevalence  gap_temporal
detector                                                                            
Ensemble         0.0407          0.1134         0.1706         -0.0727       -0.0572
OneClassSVM      0.0674          0.1399         0.1018         -0.0726        0.0382
KMeans           0.0151          0.0362         0.0458         -0.0211       -0.0096
DBSCAN           0.0066          0.0963         0.0248         -0.0897        0.0715
IsolationForest  0.0080          0.0182         0.0165         -0.0101        0.0017
LOF              0.2248          0.3005         0.0030         -0.0758        0.2975

  ULB/N decomposition:
                 random  random_matched  chronological  gap_prevalence  gap_temporal
detector                                                                            
LOF              0.6712          0.5452         0.6032   

In [52]:
# =============================================================================
# 13. PHASE 0  --  everything the revision still needs, in one cell
#     Run AFTER cells 0-4 (SPLITS, DETECTORS, calibrator, helpers) have run.
#     Self-contained otherwise: recomputes the headline scores it needs.
#
#   P0.1  paired block bootstrap  Ensemble - best single member   (cell 13 redo)
#   P0.2  hyperparameter selection with an explicit tie-break      (Table 16 / Table 3)
#   P0.3  PR-AUC / prevalence per rolling origin                   (Table 9 'lift' fix)
#   P0.4  figures regenerated: PR (full title), rolling (integer x),
#         split gap (legend moved), label efficiency (n valid draws)
# =============================================================================
import os, json, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, precision_recall_curve

P0_DIR = os.path.join(OUT_DIR, "phase0"); os.makedirs(P0_DIR, exist_ok=True)
FIG0   = os.path.join(P0_DIR, "figures");  os.makedirs(FIG0, exist_ok=True)

# ---------------------------------------------------------------------------
# Headline scores, recomputed once (same seed / rho as the 'headline' run)
# ---------------------------------------------------------------------------
print("recomputing headline scores ...")
SC = {}
for ds in ["ULB", "PaySim"]:
    sp = SPLITS[ds]
    for regime in ["U", "N"]:
        Xref = reference_set(sp, regime); rho = rho_for(sp, regime, RHO_DEFAULT)
        cals, s_te = {}, {}
        for name, fn in DETECTORS.items():
            kw = {"nu": rho} if name == "OneClassSVM" else {}
            sc = fn(Xref, sp.Xva, sp.Xte, seed=SEED, **kw)
            cals[name] = ValidationCalibrator(sc.val); s_te[name] = sc.test
        s_te["Ensemble"] = ecdf_rank_average(cals, {k: s_te[k] for k in cals})
        SC[(ds, regime)] = (sp.yte, s_te)
        # consistency with the canonical frame
        ref = results_frame()
        ref = ref[(ref.experiment == "headline") & (ref.dataset == ds)
                  & (ref.regime == regime) & (ref.metric == "PR_AUC")]
        look = dict(zip(ref.detector, ref.value))
        for k, s in s_te.items():
            ap = average_precision_score(sp.yte, s)
            assert abs(ap - look[k]) < 1e-6, f"{ds}/{regime}/{k}: {ap} vs canonical {look[k]}"
print("  scores match canonical_results.csv")

# ---------------------------------------------------------------------------
# P0.1  paired bootstrap: Ensemble vs best single member, and vs every member
# ---------------------------------------------------------------------------
rows = []
for (ds, regime), (y, s) in SC.items():
    members = {k: v for k, v in s.items() if k != "Ensemble"}
    best = max(members, key=lambda k: average_precision_score(y, members[k]))
    for rival, sv in members.items():
        d, pwin, (lo, hi) = paired_bootstrap_diff(y, s["Ensemble"], sv, n_boot=2000)
        rows.append({"dataset": ds, "regime": regime, "rival": rival,
                     "is_best_single": int(rival == best),
                     "AP_ensemble": average_precision_score(y, s["Ensemble"]),
                     "AP_rival": average_precision_score(y, sv),
                     "diff": d, "ci_lo": lo, "ci_hi": hi, "P_ensemble_ahead": pwin,
                     "verdict": ("ensemble ahead" if lo > 0 else
                                 "ensemble behind" if hi < 0 else "within noise")})
paired = pd.DataFrame(rows).round(4)
paired.to_csv(os.path.join(P0_DIR, "table_paired_bootstrap_ensemble.csv"), index=False)
print("\n=== P0.1 paired block bootstrap (2000 replicates), Ensemble - member ===")
print(paired.to_string(index=False))

# ---------------------------------------------------------------------------
# P0.2  hyperparameter selection with explicit tie-break
#       rule: maximise val_PR_AUC; ties (|diff| < 1e-4) -> simplest configuration
# ---------------------------------------------------------------------------
sens_path = os.path.join(OUT_DIR, "table_hyperparameter_sensitivity.csv")
sens = pd.read_csv(sens_path)
TIE = 1e-4

def _complexity(r):
    """Lower = simpler. Used only to break ties on validation PR-AUC."""
    if r.detector == "LOF":              return r.n_neighbors
    if r.detector == "KMeans":           return r.k
    if r.detector == "IsolationForest":  return r.n_estimators
    if r.detector == "OneClassSVM":      return {"scale": 0}.get(str(r.gamma), 1 + float(r.gamma) if str(r.gamma) not in ("nan",) else 0)
    if r.detector == "DBSCAN":
        e = r.eps_scale if pd.notna(r.eps_scale) else 1.0
        m = r.min_samples if pd.notna(r.min_samples) else 10
        return abs(np.log(e)) + abs(m - 10) / 100     # closest to the knee estimate
    return 0

sel_rows, tie_rows = [], []
for key, g in sens.groupby(["dataset", "regime", "detector"]):
    g = g.copy(); g["complexity"] = g.apply(_complexity, axis=1)
    top = g.val_PR_AUC.max()
    tied = g[g.val_PR_AUC >= top - TIE].sort_values("complexity")
    chosen = tied.iloc[0]
    sel_rows.append({"dataset": key[0], "regime": key[1], "detector": key[2],
                     "selected_config": chosen.config,
                     "val_PR_AUC": chosen.val_PR_AUC, "test_PR_AUC": chosen.test_PR_AUC,
                     "test_min": g.test_PR_AUC.min(), "test_max": g.test_PR_AUC.max(),
                     "test_spread": g.test_PR_AUC.max() - g.test_PR_AUC.min(),
                     "n_tied_on_validation": len(tied),
                     "tied_configs": " | ".join(tied.config.tolist())})
sel = pd.DataFrame(sel_rows).round(4)
sel.to_csv(os.path.join(P0_DIR, "table16_hyperparameter_selected.csv"), index=False)
print("\n=== P0.2 validation-selected configurations (tie-break: simplest) ===")
print(sel.drop(columns=["tied_configs"]).to_string(index=False))
print("\n  configurations tied on validation (|dVal| < 1e-4):")
print(sel[sel.n_tied_on_validation > 1][["dataset", "regime", "detector", "tied_configs"]]
      .to_string(index=False))

# Table 3 'selected' block: one column per (dataset, regime)
t3 = sel.pivot_table(index="detector", columns=["dataset", "regime"],
                     values="selected_config", aggfunc="first")
t3.to_csv(os.path.join(P0_DIR, "table3_selected_block.csv"))
print("\n=== Table 3, selected-configuration block ===")
print(t3.to_string())

# ---------------------------------------------------------------------------
# P0.3  rolling origins: PR-AUC / prevalence per fold (replaces 'lift')
# ---------------------------------------------------------------------------
cf = results_frame()
r = cf[(cf.experiment == "rolling_origin") & (cf.metric.isin(["PR_AUC", "prevalence"]))]
per_fold = r.pivot_table(index=["dataset", "regime", "detector", "fold"],
                         columns="metric", values="value").reset_index()
per_fold["pr_lift"] = per_fold.PR_AUC / per_fold.prevalence
per_fold = per_fold.round(4)
per_fold.to_csv(os.path.join(P0_DIR, "table9_rolling_per_fold.csv"), index=False)
agg = (per_fold.groupby(["dataset", "regime", "detector"])
       .agg(pr_mean=("PR_AUC", "mean"), pr_std=("PR_AUC", "std"),
            pr_min=("PR_AUC", "min"), pr_max=("PR_AUC", "max"),
            prlift_mean=("pr_lift", "mean"), prlift_min=("pr_lift", "min"),
            prlift_max=("pr_lift", "max"), n=("PR_AUC", "count")).reset_index())
agg["cv"] = agg.pr_std / agg.pr_mean
agg = agg.round(4)
agg.to_csv(os.path.join(P0_DIR, "table9_rolling_origin_prlift.csv"), index=False)
print("\n=== P0.3 rolling origins, PR-AUC and PR-AUC/prevalence ===")
print(agg.to_string(index=False))
print("\n  per-fold, PaySim regime N (for the Section 4.6 sentence):")
print(per_fold[(per_fold.dataset == "PaySim") & (per_fold.regime == "N")]
      .pivot_table(index="detector", columns="fold", values="pr_lift").round(1).to_string())
print("\n  per-fold, ULB regime N:")
print(per_fold[(per_fold.dataset == "ULB") & (per_fold.regime == "N")]
      .pivot_table(index="detector", columns="fold", values="PR_AUC").round(4).to_string())

# ---------------------------------------------------------------------------
# P0.4  figures
# ---------------------------------------------------------------------------
def fig_pr(ds, regime):
    y, s = SC[(ds, regime)]
    fig, ax = plt.subplots(figsize=(7, 5))
    for name, sc in s.items():
        p, rr, _ = precision_recall_curve(y, sc)
        ax.plot(rr, p, lw=1.4, label=f"{name} AP={average_precision_score(y, sc):.3f}")
    base = float(y.mean())
    ax.axhline(base, ls="--", c="grey", lw=1, label=f"baseline={base:.4f}")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title(f"{ds}, chronological test set, regime {regime}\n"
                 f"({int(y.sum())} frauds / {len(y):,} transactions)", fontsize=10)
    ax.legend(fontsize=7); fig.tight_layout()
    p = os.path.join(FIG0, f"pr_{ds}_{regime}.png"); fig.savefig(p, dpi=200); plt.close(fig)
    return p

def fig_rolling(ds, regime="N"):
    d = per_fold[(per_fold.dataset == ds) & (per_fold.regime == regime)]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    for det, g in d.groupby("detector"):
        g = g.sort_values("fold")
        axes[0].plot(g.fold + 1, g.PR_AUC, marker="o", lw=1.3, label=det)
        axes[1].plot(g.fold + 1, g.pr_lift, marker="o", lw=1.3, label=det)
    for ax, yl in zip(axes, ["Test PR-AUC", "Test PR-AUC / fold prevalence"]):
        ax.set_xticks(sorted((d.fold + 1).unique()))
        ax.set_xlabel("rolling origin (later = further into the stream)"); ax.set_ylabel(yl)
    axes[0].legend(fontsize=7)
    fig.suptitle(f"Stability across temporal origins ({ds}, regime {regime})", fontsize=11)
    fig.tight_layout()
    p = os.path.join(FIG0, f"rolling_{ds}_{regime}.png"); fig.savefig(p, dpi=200); plt.close(fig)
    return p

def fig_split_gap(ds, regime="N"):
    d = cf[(cf.experiment == "split_comparison") & (cf.dataset == ds)
           & (cf.regime == regime) & (cf.metric == "PR_AUC")]
    t = d.groupby(["detector", "split_type"]).value.mean().unstack("split_type")
    t = t.sort_values("chronological", ascending=False)
    t["gap_prevalence"] = t["random"] - t["random_matched"]
    t["gap_temporal"] = t["random_matched"] - t["chronological"]
    x = np.arange(len(t)); w = 0.27
    fig, axes = plt.subplots(2, 1, figsize=(10, 7.5))
    axes[0].bar(x - w, t["random"], w, label="random split")
    axes[0].bar(x, t["random_matched"], w, label="random, test prevalence matched")
    axes[0].bar(x + w, t["chronological"], w, label="chronological split")
    axes[0].set_ylabel("Test PR-AUC"); axes[0].set_xticks(x); axes[0].set_xticklabels(t.index, rotation=15)
    axes[0].legend(loc="upper right", fontsize=8)
    axes[0].set_title(f"Random versus chronological evaluation, {ds} (regime {regime})")
    axes[1].bar(x - w / 2, t["gap_prevalence"], w, label="attributable to test prevalence")
    axes[1].bar(x + w / 2, t["gap_temporal"], w, label="attributable to temporal ordering")
    axes[1].axhline(0, c="k", lw=0.8); axes[1].set_ylabel("PR-AUC difference")
    axes[1].set_xticks(x); axes[1].set_xticklabels(t.index, rotation=15)
    axes[1].legend(loc="best", fontsize=8, framealpha=0.95)
    axes[1].set_title("Decomposition of the gap; only the second component admits a temporal reading", fontsize=10)
    fig.tight_layout()
    p = os.path.join(FIG0, f"split_gap_{ds}_{regime}.png"); fig.savefig(p, dpi=200); plt.close(fig)
    return p

def fig_label_efficiency():
    le = pd.read_csv(os.path.join(OUT_DIR, "table_label_efficiency.csv"))
    base = float(results_frame().query(
        "experiment=='headline' and dataset=='ULB' and regime=='N' and detector=='Ensemble' and metric=='PR_AUC'").value.iloc[0])
    fig, ax = plt.subplots(figsize=(7.5, 4.6))
    x = le["budget"].values
    ax.errorbar(x, le["mean"], yerr=le["std"].fillna(0), marker="o", capsize=3,
                label="logistic meta-classifier (mean ± sd over draws)")
    ax.fill_between(x, le["lo"], le["hi"], alpha=0.15, label="2.5–97.5 % of draws")
    ax.axhline(base, ls="--", c="grey", label=f"label-free ECDF ensemble ({base:.3f}, 0 labels)")
    for xi, n, f in zip(x, le["n"], le["frauds_mean"]):
        ax.annotate(f"n={int(n)}\n~{f:.0f} frauds", (xi, 0.005), fontsize=7, ha="center", va="bottom")
    ax.set_xscale("log"); ax.set_ylim(0, max(0.3, float(le["hi"].max()) + 0.02))
    ax.set_xlabel("labelled transactions in the budget (log scale)"); ax.set_ylabel("Test PR-AUC")
    ax.set_title("Label efficiency of the logistic meta-classifier (ULB, chronological test set)", fontsize=10)
    ax.legend(fontsize=7, loc="upper right"); fig.tight_layout()
    p = os.path.join(FIG0, "label_efficiency_ULB.png"); fig.savefig(p, dpi=200); plt.close(fig)
    return p

written = []
for ds in ["ULB", "PaySim"]:
    for regime in ["U", "N"]:
        written.append(fig_pr(ds, regime))
    written.append(fig_rolling(ds)); written.append(fig_split_gap(ds))
written.append(fig_label_efficiency())

# ---------------------------------------------------------------------------
# Manifest of this cell
# ---------------------------------------------------------------------------
with open(os.path.join(P0_DIR, "phase0_manifest.json"), "w") as f:
    json.dump({"created": pd.Timestamp.now().isoformat(), "tie_break_tolerance": TIE,
               "tie_break_rule": "simplest configuration among those within 1e-4 of the best validation PR-AUC",
               "paired_bootstrap_replicates": 2000, "blocks": BOOTSTRAP_BLOCKS,
               "files": sorted(os.listdir(P0_DIR)), "figures": written}, f, indent=2)
print("\nPhase 0 outputs ->", P0_DIR)
for p in written: print("  ", p)

recomputing headline scores ...
  scores match canonical_results.csv

=== P0.1 paired block bootstrap (2000 replicates), Ensemble - member ===
dataset regime           rival  is_best_single  AP_ensemble  AP_rival    diff   ci_lo   ci_hi  P_ensemble_ahead         verdict
    ULB      U IsolationForest               0       0.0695    0.0546  0.0152 -0.0197  0.0466            0.8425    within noise
    ULB      U             LOF               0       0.0695    0.0763 -0.0125 -0.0757  0.0337            0.3540    within noise
    ULB      U     OneClassSVM               1       0.0695    0.0839 -0.0155 -0.0477  0.0127            0.1465    within noise
    ULB      U          DBSCAN               0       0.0695    0.0247  0.0471  0.0206  0.0775            1.0000  ensemble ahead
    ULB      U          KMeans               0       0.0695    0.0679  0.0017 -0.0328  0.0339            0.5575    within noise
    ULB      N IsolationForest               0       0.1400    0.0369  0.1068  0.0657  0.

ValueError: could not convert string to float: 'scale'

In [53]:
# =============================================================================
# 13b. PHASE 0, part 2  --  P0.2 / P0.3 / P0.4
#      Run in a NEW cell. Reuses SC and `paired` left in memory by the first
#      cell (P0.1 finished before the error). If SC is missing, it recomputes it.
# =============================================================================
import os, json, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, precision_recall_curve

P0_DIR = os.path.join(OUT_DIR, "phase0"); os.makedirs(P0_DIR, exist_ok=True)
FIG0   = os.path.join(P0_DIR, "figures");  os.makedirs(FIG0, exist_ok=True)

# ---- safety: recompute SC only if the first cell did not leave it in memory --
if "SC" not in globals() or len(SC) < 4:
    print("SC not found -- recomputing headline scores ...")
    SC = {}
    for ds in ["ULB", "PaySim"]:
        sp = SPLITS[ds]
        for regime in ["U", "N"]:
            Xref = reference_set(sp, regime); rho = rho_for(sp, regime, RHO_DEFAULT)
            cals, s_te = {}, {}
            for name, fn in DETECTORS.items():
                kw = {"nu": rho} if name == "OneClassSVM" else {}
                sc = fn(Xref, sp.Xva, sp.Xte, seed=SEED, **kw)
                cals[name] = ValidationCalibrator(sc.val); s_te[name] = sc.test
            s_te["Ensemble"] = ecdf_rank_average(cals, {k: s_te[k] for k in cals})
            SC[(ds, regime)] = (sp.yte, s_te)
else:
    print("using SC from the previous cell")

# ---------------------------------------------------------------------------
# P0.2  hyperparameter selection with explicit tie-break
#       rule: maximise val_PR_AUC; ties (|diff| < 1e-4) -> simplest configuration
# ---------------------------------------------------------------------------
sens = pd.read_csv(os.path.join(OUT_DIR, "table_hyperparameter_sensitivity.csv"))
TIE = 1e-4

def _num(v, default):
    try:
        return float(v) if pd.notna(v) else default
    except (TypeError, ValueError):
        return default

def _complexity(r):
    """Lower = simpler. Used only to break ties on validation PR-AUC."""
    det = r["detector"]
    if det == "LOF":             return _num(r.get("n_neighbors"), 20)
    if det == "KMeans":          return _num(r.get("k"), 8)
    if det == "IsolationForest": return _num(r.get("n_estimators"), 200)
    if det == "OneClassSVM":
        g = r.get("gamma")
        return 0.0 if (pd.isna(g) or str(g) == "scale") else 1.0 + _num(g, 0.0)
    if det == "DBSCAN":
        e = _num(r.get("eps_scale"), 1.0); m = _num(r.get("min_samples"), 10)
        return abs(np.log(e)) + abs(m - 10) / 100.0     # closest to the knee estimate
    return 0.0

sel_rows = []
for key, g in sens.groupby(["dataset", "regime", "detector"]):
    g = g.copy(); g["complexity"] = g.apply(_complexity, axis=1)
    top = g.val_PR_AUC.max()
    tied = g[g.val_PR_AUC >= top - TIE].sort_values(["complexity", "config"])
    chosen = tied.iloc[0]
    sel_rows.append({"dataset": key[0], "regime": key[1], "detector": key[2],
                     "selected_config": chosen.config,
                     "val_PR_AUC": chosen.val_PR_AUC, "test_PR_AUC": chosen.test_PR_AUC,
                     "test_min": g.test_PR_AUC.min(), "test_max": g.test_PR_AUC.max(),
                     "test_spread": g.test_PR_AUC.max() - g.test_PR_AUC.min(),
                     "n_tied_on_validation": len(tied),
                     "tied_configs": " | ".join(tied.config.astype(str).tolist())})
sel = pd.DataFrame(sel_rows).round(4)
sel.to_csv(os.path.join(P0_DIR, "table16_hyperparameter_selected.csv"), index=False)
print("\n=== P0.2 validation-selected configurations (tie-break: simplest) ===")
print(sel.drop(columns=["tied_configs"]).to_string(index=False))
print("\n  configurations tied on validation (|dVal| < 1e-4):")
tied_only = sel[sel.n_tied_on_validation > 1]
print(tied_only[["dataset", "regime", "detector", "tied_configs"]].to_string(index=False)
      if len(tied_only) else "  (none)")

t3 = sel.pivot_table(index="detector", columns=["dataset", "regime"],
                     values="selected_config", aggfunc="first")
t3.to_csv(os.path.join(P0_DIR, "table3_selected_block.csv"))
print("\n=== Table 3, selected-configuration block ===")
print(t3.to_string())

# ---------------------------------------------------------------------------
# P0.3  rolling origins: PR-AUC / prevalence per fold
# ---------------------------------------------------------------------------
cf = results_frame()
r = cf[(cf.experiment == "rolling_origin") & (cf.metric.isin(["PR_AUC", "prevalence"]))]
per_fold = r.pivot_table(index=["dataset", "regime", "detector", "fold"],
                         columns="metric", values="value").reset_index()
per_fold["pr_lift"] = per_fold.PR_AUC / per_fold.prevalence
per_fold = per_fold.round(4)
per_fold.to_csv(os.path.join(P0_DIR, "table9_rolling_per_fold.csv"), index=False)
agg = (per_fold.groupby(["dataset", "regime", "detector"])
       .agg(pr_mean=("PR_AUC", "mean"), pr_std=("PR_AUC", "std"),
            pr_min=("PR_AUC", "min"), pr_max=("PR_AUC", "max"),
            prlift_mean=("pr_lift", "mean"), prlift_min=("pr_lift", "min"),
            prlift_max=("pr_lift", "max"), n=("PR_AUC", "count")).reset_index())
agg["cv"] = agg.pr_std / agg.pr_mean
agg = agg.round(4)
agg.to_csv(os.path.join(P0_DIR, "table9_rolling_origin_prlift.csv"), index=False)
print("\n=== P0.3 rolling origins, PR-AUC and PR-AUC/prevalence ===")
print(agg.to_string(index=False))
print("\n  per-fold PR-AUC/prevalence, PaySim regime N:")
print(per_fold[(per_fold.dataset == "PaySim") & (per_fold.regime == "N")]
      .pivot_table(index="detector", columns="fold", values="pr_lift").round(1).to_string())
print("\n  per-fold PR-AUC, ULB regime N:")
print(per_fold[(per_fold.dataset == "ULB") & (per_fold.regime == "N")]
      .pivot_table(index="detector", columns="fold", values="PR_AUC").round(4).to_string())

# ---------------------------------------------------------------------------
# P0.4  figures
# ---------------------------------------------------------------------------
def fig_pr(ds, regime):
    y, s = SC[(ds, regime)]
    fig, ax = plt.subplots(figsize=(7, 5))
    for name, sc in s.items():
        p, rr, _ = precision_recall_curve(y, sc)
        ax.plot(rr, p, lw=1.4, label=f"{name} AP={average_precision_score(y, sc):.3f}")
    base = float(np.mean(y))
    ax.axhline(base, ls="--", c="grey", lw=1, label=f"baseline={base:.4f}")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title(f"{ds}, chronological test set, regime {regime}\n"
                 f"({int(np.sum(y))} frauds / {len(y):,} transactions)", fontsize=10)
    ax.legend(fontsize=7); fig.tight_layout()
    p = os.path.join(FIG0, f"pr_{ds}_{regime}.png"); fig.savefig(p, dpi=200); plt.close(fig)
    return p

def fig_rolling(ds, regime="N"):
    d = per_fold[(per_fold.dataset == ds) & (per_fold.regime == regime)]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    for det, g in d.groupby("detector"):
        g = g.sort_values("fold")
        axes[0].plot(g.fold + 1, g.PR_AUC, marker="o", lw=1.3, label=det)
        axes[1].plot(g.fold + 1, g.pr_lift, marker="o", lw=1.3, label=det)
    for ax, yl in zip(axes, ["Test PR-AUC", "Test PR-AUC / fold prevalence"]):
        ax.set_xticks(sorted((d.fold + 1).unique()))
        ax.set_xlabel("rolling origin (later = further into the stream)"); ax.set_ylabel(yl)
    axes[0].legend(fontsize=7)
    fig.suptitle(f"Stability across temporal origins ({ds}, regime {regime})", fontsize=11)
    fig.tight_layout()
    p = os.path.join(FIG0, f"rolling_{ds}_{regime}.png"); fig.savefig(p, dpi=200); plt.close(fig)
    return p

def fig_split_gap(ds, regime="N"):
    d = cf[(cf.experiment == "split_comparison") & (cf.dataset == ds)
           & (cf.regime == regime) & (cf.metric == "PR_AUC")]
    t = d.groupby(["detector", "split_type"]).value.mean().unstack("split_type")
    t = t.sort_values("chronological", ascending=False)
    t["gap_prevalence"] = t["random"] - t["random_matched"]
    t["gap_temporal"] = t["random_matched"] - t["chronological"]
    x = np.arange(len(t)); w = 0.27
    fig, axes = plt.subplots(2, 1, figsize=(10, 7.5))
    axes[0].bar(x - w, t["random"], w, label="random split")
    axes[0].bar(x, t["random_matched"], w, label="random, test prevalence matched")
    axes[0].bar(x + w, t["chronological"], w, label="chronological split")
    axes[0].set_ylabel("Test PR-AUC"); axes[0].set_xticks(x); axes[0].set_xticklabels(t.index, rotation=15)
    axes[0].legend(loc="upper right", fontsize=8)
    axes[0].set_title(f"Random versus chronological evaluation, {ds} (regime {regime})")
    axes[1].bar(x - w / 2, t["gap_prevalence"], w, label="attributable to test prevalence")
    axes[1].bar(x + w / 2, t["gap_temporal"], w, label="attributable to temporal ordering")
    axes[1].axhline(0, c="k", lw=0.8); axes[1].set_ylabel("PR-AUC difference")
    axes[1].set_xticks(x); axes[1].set_xticklabels(t.index, rotation=15)
    axes[1].legend(loc="best", fontsize=8, framealpha=0.95)
    axes[1].set_title("Decomposition of the gap; only the second component admits a temporal reading", fontsize=10)
    fig.tight_layout()
    p = os.path.join(FIG0, f"split_gap_{ds}_{regime}.png"); fig.savefig(p, dpi=200); plt.close(fig)
    return p

def fig_label_efficiency():
    le = pd.read_csv(os.path.join(OUT_DIR, "table_label_efficiency.csv"))
    base = float(cf.query("experiment=='headline' and dataset=='ULB' and regime=='N' "
                          "and detector=='Ensemble' and metric=='PR_AUC'").value.iloc[0])
    x = le["budget"].values
    fig, ax = plt.subplots(figsize=(7.5, 4.6))
    ax.errorbar(x, le["mean"], yerr=le["std"].fillna(0), marker="o", capsize=3,
                label="logistic meta-classifier (mean ± sd over draws)")
    if {"lo", "hi"} <= set(le.columns):
        ax.fill_between(x, le["lo"], le["hi"], alpha=0.15, label="2.5–97.5 % of draws")
    ax.axhline(base, ls="--", c="grey", label=f"label-free ECDF ensemble ({base:.3f}, 0 labels)")
    if {"n", "frauds_mean"} <= set(le.columns):
        for xi, n, f in zip(x, le["n"], le["frauds_mean"]):
            ax.annotate(f"n={int(n)}\n~{f:.0f} frauds", (xi, 0.005), fontsize=7, ha="center", va="bottom")
    ax.set_xscale("log")
    ymax = float(le["hi"].max()) if "hi" in le.columns else float((le["mean"] + le["std"].fillna(0)).max())
    ax.set_ylim(0, max(0.3, ymax + 0.02))
    ax.set_xlabel("labelled transactions in the budget (log scale)"); ax.set_ylabel("Test PR-AUC")
    ax.set_title("Label efficiency of the logistic meta-classifier (ULB, chronological test set)", fontsize=10)
    ax.legend(fontsize=7, loc="upper right"); fig.tight_layout()
    p = os.path.join(FIG0, "label_efficiency_ULB.png"); fig.savefig(p, dpi=200); plt.close(fig)
    return p

written = []
for ds in ["ULB", "PaySim"]:
    for regime in ["U", "N"]:
        written.append(fig_pr(ds, regime))
    written.append(fig_rolling(ds)); written.append(fig_split_gap(ds))
written.append(fig_label_efficiency())

# ---------------------------------------------------------------------------
# Manifest
# ---------------------------------------------------------------------------
with open(os.path.join(P0_DIR, "phase0_manifest.json"), "w") as f:
    json.dump({"created": pd.Timestamp.now().isoformat(), "tie_break_tolerance": TIE,
               "tie_break_rule": "simplest configuration among those within 1e-4 of the best validation PR-AUC",
               "paired_bootstrap_replicates": 2000,
               "blocks": globals().get("BOOTSTRAP_BLOCKS", None),
               "files": sorted(os.listdir(P0_DIR)), "figures": written}, f, indent=2)
print("\nPhase 0 outputs ->", P0_DIR)
for p in written: print("  ", p)

using SC from the previous cell

=== P0.2 validation-selected configurations (tie-break: simplest) ===
dataset regime        detector       selected_config  val_PR_AUC  test_PR_AUC  test_min  test_max  test_spread  n_tied_on_validation
 PaySim      N          DBSCAN   {"min_samples": 25}      0.0191       0.2169    0.0663    0.2169       0.1506                     3
 PaySim      N IsolationForest {"n_estimators": 200}      0.0021       0.0687    0.0512    0.0709       0.0197                     2
 PaySim      N          KMeans             {"k": 32}      0.0114       0.1378    0.0750    0.1378       0.0628                     1
 PaySim      N             LOF    {"n_neighbors": 5}      0.0004       0.0068    0.0068    0.0078       0.0010                     5
 PaySim      N     OneClassSVM        {"gamma": 0.1}      0.0226       0.2164    0.1611    0.2185       0.0574                     1
 PaySim      U          DBSCAN    {"eps_scale": 0.5}      0.0192       0.2219    0.0663    0.2219  

In [54]:
# PHASE 0, part 3 -- PR and ROC figures sized for a 3.4-inch column
import os, numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve, average_precision_score, roc_auc_score
FIG3 = os.path.join(OUT_DIR, "phase0", "figures_column"); os.makedirs(FIG3, exist_ok=True)
plt.rcParams.update({"font.size": 13, "axes.titlesize": 13, "axes.labelsize": 13,
                     "legend.fontsize": 11, "xtick.labelsize": 12, "ytick.labelsize": 12})

def col_fig(ds, regime, kind):
    y, s = SC[(ds, regime)]
    fig, ax = plt.subplots(figsize=(6.8, 5.6))        # 2x the column width -> halves cleanly
    for name, sc in s.items():
        if kind == "pr":
            p, r, _ = precision_recall_curve(y, sc)
            ax.plot(r, p, lw=1.6, label=f"{name}  AP={average_precision_score(y, sc):.3f}")
        else:
            fpr, tpr, _ = roc_curve(y, sc)
            ax.plot(fpr, tpr, lw=1.6, label=f"{name}  AUC={roc_auc_score(y, sc):.3f}")
    if kind == "pr":
        base = float(np.mean(y)); ax.axhline(base, ls="--", c="grey", lw=1, label=f"baseline={base:.4f}")
        ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_title(f"{ds}, regime {regime}: precision-recall")
    else:
        ax.plot([0, 1], [0, 1], ls="--", c="grey", lw=1)
        ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
        ax.set_title(f"{ds}, regime {regime}: ROC (secondary metric)")
    ax.legend(loc="lower right" if kind == "roc" else "upper right", frameon=True)
    fig.tight_layout()
    p = os.path.join(FIG3, f"{kind}_{ds}_{regime}.png"); fig.savefig(p, dpi=300); plt.close(fig); return p

for ds in ["ULB", "PaySim"]:
    for regime in ["U", "N"]:
        for kind in ["pr", "roc"]:
            print(col_fig(ds, regime, kind))

revision_v2\phase0\figures_column\pr_ULB_U.png
revision_v2\phase0\figures_column\roc_ULB_U.png
revision_v2\phase0\figures_column\pr_ULB_N.png
revision_v2\phase0\figures_column\roc_ULB_N.png
revision_v2\phase0\figures_column\pr_PaySim_U.png
revision_v2\phase0\figures_column\roc_PaySim_U.png
revision_v2\phase0\figures_column\pr_PaySim_N.png
revision_v2\phase0\figures_column\roc_PaySim_N.png


In [55]:
# PHASE 0, part 4 -- true column template (3.0 in, native 8 pt)
import os, numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve, average_precision_score, roc_auc_score
FIG4 = os.path.join(OUT_DIR, "phase0", "figures_col3"); os.makedirs(FIG4, exist_ok=True)
plt.rcParams.update({"font.size": 8, "axes.titlesize": 8.5, "axes.labelsize": 8,
                     "legend.fontsize": 7, "xtick.labelsize": 7.5, "ytick.labelsize": 7.5,
                     "axes.linewidth": 0.6, "lines.linewidth": 1.1})
SHORT = {"IsolationForest": "IF", "OneClassSVM": "OC-SVM", "KMeans": "K-Means",
         "LOF": "LOF", "DBSCAN": "DBSCAN", "Ensemble": "Ensemble"}

def col3(ds, regime, kind):
    y, s = SC[(ds, regime)]
    fig, ax = plt.subplots(figsize=(3.0, 3.1))
    for name, sc in s.items():
        if kind == "pr":
            p, r, _ = precision_recall_curve(y, sc); ax.plot(r, p, label=f"{SHORT[name]} {average_precision_score(y, sc):.3f}")
        else:
            f, t, _ = roc_curve(y, sc); ax.plot(f, t, label=f"{SHORT[name]} {roc_auc_score(y, sc):.3f}")
    if kind == "pr":
        base = float(np.mean(y)); ax.axhline(base, ls="--", c="grey", lw=0.8, label=f"baseline {base:.4f}")
        ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_title(f"{ds}, regime {regime}: PR (AP in legend)")
    else:
        ax.plot([0, 1], [0, 1], ls="--", c="grey", lw=0.8)
        ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
        ax.set_title(f"{ds}, regime {regime}: ROC (AUC in legend)")
    ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2, frameon=False,
              handlelength=1.4, columnspacing=1.0)
    fig.subplots_adjust(left=0.17, right=0.98, top=0.90, bottom=0.36)
    p = os.path.join(FIG4, f"{kind}_{ds}_{regime}.png"); fig.savefig(p, dpi=600); plt.close(fig); return p

for ds in ["ULB", "PaySim"]:
    for regime in ["U", "N"]:
        for kind in ["pr", "roc"]:
            print(col3(ds, regime, kind))

revision_v2\phase0\figures_col3\pr_ULB_U.png
revision_v2\phase0\figures_col3\roc_ULB_U.png
revision_v2\phase0\figures_col3\pr_ULB_N.png
revision_v2\phase0\figures_col3\roc_ULB_N.png
revision_v2\phase0\figures_col3\pr_PaySim_U.png
revision_v2\phase0\figures_col3\roc_PaySim_U.png
revision_v2\phase0\figures_col3\pr_PaySim_N.png
revision_v2\phase0\figures_col3\roc_PaySim_N.png
